STAGE 4.4: RESNET18 SUPERVISED TRAINING

In [28]:
# =============================================================================
# CISLR — STAGE 4.4 RESNET18 v2
#
# IMPROVED CONVERGENCE / TRAINING VERSION
#
# IMPORTANT:
# - Stage 4.1 READ ONLY
# - Stage 4.2 READ ONLY
# - Stage 4.3 READ ONLY
# - Existing Stage 4.4 common pool READ ONLY
# - Existing frame cache READ ONLY
# - NO VIDEO RE-DECODING
#
# Main improvements over v1:
#
# 1. VIDEO-LEVEL TRAINING
#    8 frame logits are averaged before CrossEntropyLoss.
#
# 2. NO HORIZONTAL FLIP
#    Avoids potentially changing ISL handedness/spatial semantics.
#
# 3. LIGHTER AUGMENTATION
#    Resize + small affine instead of aggressive RandomResizedCrop.
#
# 4. CLASSIFIER WARM-UP
#    First epoch trains only the randomly initialized classifier.
#
# 5. DISCRIMINATIVE LEARNING RATES
#    Backbone: 1e-5
#    Classifier: 5e-4
#
# 6. COSINE LR DECAY
#
# 7. GRADIENT CLIPPING
#
# 8. TOP-1 + TOP-5 TRAINING DIAGNOSTICS
#
# 9. 25 EPOCHS
#
# This is still a TRAINING DIAGNOSTIC.
# Official CISLR evaluation remains prototype -> test retrieval.
# =============================================================================


import os
import gc
import json
import time
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

import torch
import torch.nn as nn

from torch.utils.data import Dataset, DataLoader

from torchvision import models, transforms

from PIL import Image

warnings.filterwarnings("ignore")


# =============================================================================
# 1. PATHS
# =============================================================================

PROJECT_ROOT = Path(
    "/home/dipshikha/Music/cao"
)

STAGE44_DIR = (
    PROJECT_ROOT
    / "CISLR_RESEARCH"
    / "audit"
    / "stage4_4_training"
)

MODEL_DIR = (
    PROJECT_ROOT
    / "CISLR_RESEARCH"
    / "models"
    / "stage4_4"
)

CACHE_DIR = (
    STAGE44_DIR
    / "frame_cache"
)

COMMON_POOL_PATH = (
    STAGE44_DIR
    / "stage4_4_common_readable_uid_pool.csv"
)

if not COMMON_POOL_PATH.exists():

    raise FileNotFoundError(
        f"""
Existing Stage 4.4 common pool not found:

{COMMON_POOL_PATH}

Do NOT rebuild it automatically.
Check the previous Stage 4.4 run first.
"""
    )

if not CACHE_DIR.exists():

    raise FileNotFoundError(
        f"""
Existing Stage 4.4 frame cache not found:

{CACHE_DIR}

Do NOT rebuild it automatically.
"""
    )

MODEL_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# =============================================================================
# 2. CONFIGURATION
# =============================================================================

SEED = 42

IMAGE_SIZE = 224
NUM_FRAMES = 8
NUM_CLASSES = 4764

EPOCHS = 25

# ResNet18 fits comfortably with this.
BATCH_SIZE = 16

NUM_WORKERS = 2

USE_AMP = torch.cuda.is_available()

DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

# -------------------------------------------------------------
# Optimization
# -------------------------------------------------------------

BACKBONE_LR = 1e-5

CLASSIFIER_LR = 5e-4

WEIGHT_DECAY = 1e-4

WARMUP_EPOCHS = 1

GRAD_CLIP = 1.0


# =============================================================================
# 3. REPRODUCIBILITY
# =============================================================================

def seed_everything(seed):

    random.seed(seed)

    np.random.seed(seed)

    torch.manual_seed(seed)

    if torch.cuda.is_available():

        torch.cuda.manual_seed(seed)

        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True

    torch.backends.cudnn.benchmark = False


seed_everything(SEED)


# =============================================================================
# 4. CLEAN CUDA MEMORY
# =============================================================================

gc.collect()

if torch.cuda.is_available():

    torch.cuda.empty_cache()


# =============================================================================
# 5. HEADER
# =============================================================================

print("=" * 90)
print("CISLR — STAGE 4.4 RESNET18 v2")
print("=" * 90)

print(
    f"Device              : {DEVICE}"
)

if torch.cuda.is_available():

    print(
        f"GPU                 : "
        f"{torch.cuda.get_device_name(0)}"
    )

    total_vram = (
        torch.cuda.get_device_properties(0).total_memory
        / (1024 ** 3)
    )

    print(
        f"GPU VRAM            : "
        f"{total_vram:.2f} GB"
    )

print(
    f"Classes             : {NUM_CLASSES:,}"
)

print(
    f"Frames/video        : {NUM_FRAMES}"
)

print(
    f"Image size          : {IMAGE_SIZE}x{IMAGE_SIZE}"
)

print(
    f"Batch size          : {BATCH_SIZE}"
)

print(
    f"Epochs              : {EPOCHS}"
)

print(
    f"Backbone LR         : {BACKBONE_LR}"
)

print(
    f"Classifier LR       : {CLASSIFIER_LR}"
)

print(
    f"Weight decay        : {WEIGHT_DECAY}"
)

print(
    f"Warmup epochs       : {WARMUP_EPOCHS}"
)

print(
    f"Gradient clipping   : {GRAD_CLIP}"
)

print(
    f"AMP                 : {USE_AMP}"
)

print()
print(
    "Existing Stage 4.4 cache will be reused."
)


# =============================================================================
# 6. LOAD FROZEN COMMON POOL
# =============================================================================

print()
print("=" * 90)
print("LOADING FROZEN COMMON READABLE POOL")
print("=" * 90)

common_pool = pd.read_csv(
    COMMON_POOL_PATH
)

required_pool_columns = [
    "uid",
    "gloss",
    "class_id",
    "video_path",
    "frame_indices",
    "cache_path",
    "status",
]

missing_pool_columns = [
    c
    for c in required_pool_columns
    if c not in common_pool.columns
]

if missing_pool_columns:

    raise RuntimeError(
        "Common pool missing columns:\n"
        + "\n".join(missing_pool_columns)
    )


common_pool = (
    common_pool[
        common_pool["status"] == "OK"
    ]
    .copy()
    .reset_index(drop=True)
)


print(
    f"Readable videos     : "
    f"{len(common_pool):,}"
)

print(
    f"Unique classes      : "
    f"{common_pool['class_id'].nunique():,}"
)


# =============================================================================
# 7. STRONG INTEGRITY CHECKS
# =============================================================================

print()
print("=" * 90)
print("VERIFYING COMMON POOL")
print("=" * 90)


# One video per UID
if common_pool["uid"].duplicated().any():

    duplicates = (
        common_pool[
            common_pool["uid"].duplicated(
                keep=False
            )
        ]
    )

    raise RuntimeError(
        "Duplicate UID found in common pool:\n"
        + str(
            duplicates.head(20)
        )
    )


# One class per UID
uid_class_counts = (
    common_pool
    .groupby("uid")["class_id"]
    .nunique()
)

if (uid_class_counts > 1).any():

    raise RuntimeError(
        "Some UID maps to multiple class IDs."
    )


# One UID per class
class_uid_counts = (
    common_pool
    .groupby("class_id")["uid"]
    .nunique()
)

duplicate_class_uids = (
    class_uid_counts[
        class_uid_counts > 1
    ]
)

if len(duplicate_class_uids) > 0:

    print(
        "WARNING:"
    )

    print(
        f"{len(duplicate_class_uids)} classes "
        "have multiple UIDs in readable pool."
    )

    print(
        duplicate_class_uids.head(20)
    )


# Class ID range
if (
    common_pool["class_id"].min() < 0
    or
    common_pool["class_id"].max() >= NUM_CLASSES
):

    raise RuntimeError(
        "Class IDs outside expected range."
    )


# Cache existence
missing_cache = []

for p in common_pool["cache_path"]:

    if not Path(p).exists():

        missing_cache.append(p)

        if len(missing_cache) >= 20:

            break


if missing_cache:

    raise RuntimeError(
        "Missing cached frame files.\n"
        + "\n".join(missing_cache)
    )


print(
    "Pool integrity checks PASSED."
)


# =============================================================================
# 8. VERIFY CACHE CONTENT
# =============================================================================

print()
print("=" * 90)
print("VERIFYING CACHED FRAME FORMAT")
print("=" * 90)


def validate_cache_file(
    path
):

    try:

        with np.load(
            path,
            allow_pickle=False
        ) as data:

            if "frames" not in data:

                return False

            frames = data["frames"]

            expected_shape = (
                NUM_FRAMES,
                IMAGE_SIZE,
                IMAGE_SIZE,
                3
            )

            if frames.shape != expected_shape:

                return False

            if frames.dtype != np.uint8:

                return False

        return True

    except Exception:

        return False


cache_check_count = min(
    100,
    len(common_pool)
)

cache_samples = common_pool.iloc[
    np.linspace(
        0,
        len(common_pool) - 1,
        cache_check_count
    ).astype(int)
]

bad_cache = []

for _, row in cache_samples.iterrows():

    if not validate_cache_file(
        row["cache_path"]
    ):

        bad_cache.append(
            row["uid"]
        )

if bad_cache:

    raise RuntimeError(
        "Invalid cache files found during sample verification:\n"
        + "\n".join(
            map(str, bad_cache[:20])
        )
    )


print(
    f"Verified {cache_check_count} "
    "representative cache files."
)

print(
    "Cache format PASSED."
)


# =============================================================================
# 9. DATASET
# =============================================================================

class CachedCISLRDataset(
    Dataset
):

    def __init__(
        self,
        dataframe,
        transform=None
    ):

        self.df = (
            dataframe
            .reset_index(drop=True)
        )

        self.transform = transform


    def __len__(self):

        return len(self.df)


    def __getitem__(
        self,
        index
    ):

        row = self.df.iloc[index]

        cache_path = row["cache_path"]

        try:

            with np.load(
                cache_path,
                allow_pickle=False
            ) as data:

                frames = data["frames"]

        except Exception as e:

            raise RuntimeError(
                f"""
Could not load cached frames.

UID:
{row['uid']}

Cache:
{cache_path}

Error:
{e}
"""
            )


        if frames.shape != (
            NUM_FRAMES,
            IMAGE_SIZE,
            IMAGE_SIZE,
            3
        ):

            raise RuntimeError(
                f"""
Unexpected cache shape.

UID:
{row['uid']}

Shape:
{frames.shape}
"""
            )


        transformed_frames = []


        for frame in frames:

            image = Image.fromarray(
                frame
            )

            if self.transform is not None:

                image = self.transform(
                    image
                )

            transformed_frames.append(
                image
            )


        # [8, 3, 224, 224]
        frames_tensor = torch.stack(
            transformed_frames,
            dim=0
        )


        label = torch.tensor(
            int(row["class_id"]),
            dtype=torch.long
        )


        return (
            frames_tensor,
            label
        )


# =============================================================================
# 10. TRAINING TRANSFORM
#
# IMPORTANT:
# No horizontal flip.
#
# We also avoid aggressive RandomResizedCrop because the immediate goal
# is to establish whether ResNet18 can actually converge.
# =============================================================================

train_transform = transforms.Compose([

    transforms.Resize(
        (
            IMAGE_SIZE,
            IMAGE_SIZE
        )
    ),

    transforms.RandomAffine(
        degrees=5,
        translate=(
            0.05,
            0.05
        ),
        scale=(
            0.95,
            1.05
        )
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[
            0.485,
            0.456,
            0.406
        ],
        std=[
            0.229,
            0.224,
            0.225
        ]
    )

])


# =============================================================================
# 11. DATALOADER
# =============================================================================

train_dataset = CachedCISLRDataset(
    common_pool,
    transform=train_transform
)


train_loader = DataLoader(

    train_dataset,

    batch_size=BATCH_SIZE,

    shuffle=True,

    num_workers=NUM_WORKERS,

    pin_memory=torch.cuda.is_available(),

    persistent_workers=(
        NUM_WORKERS > 0
    ),

    drop_last=False

)


print()
print("=" * 90)
print("TRAINING DATA READY")
print("=" * 90)

print(
    f"Videos              : "
    f"{len(train_dataset):,}"
)

print(
    f"Frames/video        : "
    f"{NUM_FRAMES}"
)

print(
    f"Total frame samples : "
    f"{len(train_dataset) * NUM_FRAMES:,}"
)

print(
    f"Batches/epoch       : "
    f"{len(train_loader):,}"
)


# =============================================================================
# 12. BUILD PRETRAINED RESNET18
# =============================================================================

print()
print("=" * 90)
print("BUILDING RESNET18")
print("=" * 90)


weights = (
    models.ResNet18_Weights.DEFAULT
)


model = models.resnet18(
    weights=weights
)


# Replace ImageNet classifier
model.fc = nn.Linear(
    model.fc.in_features,
    NUM_CLASSES
)


model = model.to(
    DEVICE
)


total_params = sum(
    p.numel()
    for p in model.parameters()
)

trainable_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)


print(
    f"Parameters          : "
    f"{total_params:,}"
)

print(
    f"Trainable parameters: "
    f"{trainable_params:,}"
)


# =============================================================================
# 13. LOSS
# =============================================================================

criterion = nn.CrossEntropyLoss()


# =============================================================================
# 14. DISCRIMINATIVE OPTIMIZER
# =============================================================================

backbone_parameters = []

classifier_parameters = []


for name, parameter in model.named_parameters():

    if name.startswith("fc."):

        classifier_parameters.append(
            parameter
        )

    else:

        backbone_parameters.append(
            parameter
        )


optimizer = torch.optim.AdamW(

    [
        {
            "params": backbone_parameters,
            "lr": BACKBONE_LR
        },

        {
            "params": classifier_parameters,
            "lr": CLASSIFIER_LR
        }
    ],

    weight_decay=WEIGHT_DECAY

)


# =============================================================================
# 15. COSINE LR SCHEDULER
# =============================================================================

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(

    optimizer,

    T_max=max(
        1,
        EPOCHS - WARMUP_EPOCHS
    ),

    eta_min=1e-7

)


# =============================================================================
# 16. AMP
# =============================================================================

if USE_AMP:

    scaler = torch.cuda.amp.GradScaler()

else:

    scaler = None


# =============================================================================
# 17. CLASSIFIER WARM-UP CONTROL
# =============================================================================

def set_backbone_trainable(
    model,
    trainable
):

    for name, parameter in model.named_parameters():

        if name.startswith("fc."):

            parameter.requires_grad = True

        else:

            parameter.requires_grad = trainable


# During epoch 1:
# backbone frozen
#
# Epoch 2 onward:
# complete network trainable.

set_backbone_trainable(
    model,
    trainable=False
)


# =============================================================================
# 18. ARTIFACT PATHS
# =============================================================================

BEST_CHECKPOINT = (
    MODEL_DIR /
    "resnet18_stage4_4_v2_best.pt"
)

FINAL_CHECKPOINT = (
    MODEL_DIR /
    "resnet18_stage4_4_v2_final.pt"
)

HISTORY_PATH = (
    STAGE44_DIR /
    "resnet18_stage4_4_v2_training_history.csv"
)

CONFIG_PATH = (
    STAGE44_DIR /
    "resnet18_stage4_4_v2_config.json"
)


# =============================================================================
# 19. CONFIG
# =============================================================================

config = {

    "stage": "4.4",

    "version": "v2",

    "model": "ResNet18",

    "dataset": "CISLR v1.5-a",

    "seed": SEED,

    "num_classes": NUM_CLASSES,

    "image_size": IMAGE_SIZE,

    "frames_per_video": NUM_FRAMES,

    "epochs": EPOCHS,

    "batch_size": BATCH_SIZE,

    "backbone_lr": BACKBONE_LR,

    "classifier_lr": CLASSIFIER_LR,

    "weight_decay": WEIGHT_DECAY,

    "warmup_epochs": WARMUP_EPOCHS,

    "gradient_clip": GRAD_CLIP,

    "amp": USE_AMP,

    "aggregation": "mean_frame_logits",

    "horizontal_flip": False,

    "augmentation": {

        "Resize": [
            IMAGE_SIZE,
            IMAGE_SIZE
        ],

        "RandomAffine": {

            "degrees": 5,

            "translate": [
                0.05,
                0.05
            ],

            "scale": [
                0.95,
                1.05
            ]

        }

    },

    "common_pool": str(
        COMMON_POOL_PATH
    ),

    "frame_cache": str(
        CACHE_DIR
    ),

    "best_checkpoint": str(
        BEST_CHECKPOINT
    ),

    "final_checkpoint": str(
        FINAL_CHECKPOINT
    )

}


with open(
    CONFIG_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        config,
        f,
        indent=2
    )


# =============================================================================
# 20. TRAINING FUNCTION
# =============================================================================

def calculate_topk(
    logits,
    labels,
    k=5
):

    max_k = min(
        k,
        logits.size(1)
    )

    _, predictions = (
        logits.topk(
            max_k,
            dim=1
        )
    )

    correct = (
        predictions
        .eq(
            labels.unsqueeze(1)
        )
    )

    top1 = (
        correct[:, :1]
        .any(dim=1)
        .float()
        .sum()
        .item()
    )

    top5 = (
        correct[:, :max_k]
        .any(dim=1)
        .float()
        .sum()
        .item()
    )

    return top1, top5


# =============================================================================
# 21. TRAINING
# =============================================================================

print()
print("=" * 90)
print("STARTING RESNET18 v2 TRAINING")
print("=" * 90)

print()
print(
    "Epoch 1:"
)

print(
    "  Classifier warm-up"
)

print(
    "Epoch 2+:"
)

print(
    "  Full-network fine-tuning"
)

print()
print(
    "Video-level prediction:"
)

print(
    "  8 frame logits -> mean -> class prediction"
)


history = []

best_top1 = -1.0
best_top5 = -1.0
best_epoch = -1

training_start = time.time()


for epoch in range(
    1,
    EPOCHS + 1
):


    # -----------------------------------------------------------------
    # Unfreeze after warm-up
    # -----------------------------------------------------------------

    if epoch == WARMUP_EPOCHS + 1:

        set_backbone_trainable(
            model,
            trainable=True
        )

        print()
        print(
            "Backbone UNFROZEN — "
            "full-network fine-tuning begins."
        )


    model.train()


    epoch_start = time.time()


    running_loss = 0.0

    correct_top1 = 0.0

    correct_top5 = 0.0

    total_videos = 0


    num_batches = len(
        train_loader
    )


    print()
    print(
        f"-- Epoch {epoch:02d}/{EPOCHS} "
        f"starting --"
    )


    for batch_idx, (
        frames,
        labels
    ) in enumerate(
        train_loader,
        start=1
    ):


        # -------------------------------------------------------------
        # frames:
        #
        # [B, 8, 3, 224, 224]
        # -------------------------------------------------------------

        frames = frames.to(
            DEVICE,
            non_blocking=True
        )

        labels = labels.to(
            DEVICE,
            non_blocking=True
        )


        B = frames.shape[0]


        # -------------------------------------------------------------
        # Flatten frames for ResNet
        #
        # [B, 8, 3, 224, 224]
        #
        # ->
        #
        # [B*8, 3, 224, 224]
        # -------------------------------------------------------------

        frame_input = frames.reshape(

            B * NUM_FRAMES,

            3,

            IMAGE_SIZE,

            IMAGE_SIZE

        )


        optimizer.zero_grad(
            set_to_none=True
        )


        # -------------------------------------------------------------
        # Forward
        # -------------------------------------------------------------

        if USE_AMP:

            with torch.cuda.amp.autocast():

                frame_logits = model(
                    frame_input
                )

                # -----------------------------------------------------
                # [B*8, C]
                #
                # ->
                #
                # [B, 8, C]
                # -----------------------------------------------------

                frame_logits = frame_logits.reshape(
                    B,
                    NUM_FRAMES,
                    NUM_CLASSES
                )


                # -----------------------------------------------------
                # Video-level aggregation
                # -----------------------------------------------------

                video_logits = (
                    frame_logits
                    .mean(dim=1)
                )


                loss = criterion(
                    video_logits,
                    labels
                )


            scaler.scale(
                loss
            ).backward()


            scaler.unscale_(
                optimizer
            )


            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                GRAD_CLIP
            )


            scaler.step(
                optimizer
            )


            scaler.update()


        else:

            frame_logits = model(
                frame_input
            )


            frame_logits = frame_logits.reshape(
                B,
                NUM_FRAMES,
                NUM_CLASSES
            )


            video_logits = (
                frame_logits
                .mean(dim=1)
            )


            loss = criterion(
                video_logits,
                labels
            )


            loss.backward()


            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                GRAD_CLIP
            )


            optimizer.step()


        # -------------------------------------------------------------
        # Metrics
        # -------------------------------------------------------------

        top1_batch, top5_batch = (
            calculate_topk(
                video_logits,
                labels,
                k=5
            )
        )


        correct_top1 += top1_batch

        correct_top5 += top5_batch

        total_videos += B


        running_loss += (
            loss.item() * B
        )


        # -------------------------------------------------------------
        # Progress
        # -------------------------------------------------------------

        progress_interval = max(
            1,
            num_batches // 20
        )


        if (

            batch_idx == 1

            or

            batch_idx % progress_interval == 0

            or

            batch_idx == num_batches

        ):


            elapsed = (
                time.time()
                -
                epoch_start
            )


            rate = (
                batch_idx
                /
                max(
                    elapsed,
                    1e-6
                )
            )


            remaining = (
                num_batches
                -
                batch_idx
            )


            eta_seconds = (
                remaining
                /
                max(
                    rate,
                    1e-6
                )
            )


            current_loss = (
                running_loss
                /
                max(
                    total_videos,
                    1
                )
            )


            current_top1 = (
                correct_top1
                /
                max(
                    total_videos,
                    1
                )
            )


            current_top5 = (
                correct_top5
                /
                max(
                    total_videos,
                    1
                )
            )


            print(

                f"  batch "
                f"{batch_idx:,}/"
                f"{num_batches:,}"

                f" | loss "
                f"{current_loss:.4f}"

                f" | Top1 "
                f"{current_top1*100:.2f}%"

                f" | Top5 "
                f"{current_top5*100:.2f}%"

                f" | ETA "
                f"{eta_seconds/60:.1f} min"

            )


    # =================================================================
    # END EPOCH
    # =================================================================

    epoch_seconds = (
        time.time()
        -
        epoch_start
    )


    epoch_loss = (
        running_loss
        /
        max(
            total_videos,
            1
        )
    )


    epoch_top1 = (
        correct_top1
        /
        max(
            total_videos,
            1
        )
    )


    epoch_top5 = (
        correct_top5
        /
        max(
            total_videos,
            1
        )
    )


    # -----------------------------------------------------------------
    # Scheduler
    #
    # Keep LR constant during warm-up.
    # Start cosine schedule after warm-up.
    # -----------------------------------------------------------------

    if epoch > WARMUP_EPOCHS:

        scheduler.step()


    # -----------------------------------------------------------------
    # Current learning rates
    # -----------------------------------------------------------------

    backbone_current_lr = (
        optimizer.param_groups[0]["lr"]
    )

    classifier_current_lr = (
        optimizer.param_groups[1]["lr"]
    )


    # -----------------------------------------------------------------
    # Best model
    # -----------------------------------------------------------------

    is_best = (
        epoch_top1 > best_top1
    )


    if is_best:

        best_top1 = epoch_top1

        best_top5 = epoch_top5

        best_epoch = epoch


        torch.save(

            {

                "stage": "4.4",

                "version": "v2",

                "model": "ResNet18",

                "epoch": epoch,

                "model_state_dict":
                    model.state_dict(),

                "optimizer_state_dict":
                    optimizer.state_dict(),

                "train_loss":
                    epoch_loss,

                "train_top1":
                    epoch_top1,

                "train_top5":
                    epoch_top5,

                "best_train_top1":
                    best_top1,

                "best_train_top5":
                    best_top5,

                "best_epoch":
                    best_epoch,

                "seed":
                    SEED,

                "num_classes":
                    NUM_CLASSES,

                "image_size":
                    IMAGE_SIZE,

                "frames_per_video":
                    NUM_FRAMES,

                "batch_size":
                    BATCH_SIZE,

                "backbone_lr":
                    BACKBONE_LR,

                "classifier_lr":
                    CLASSIFIER_LR,

                "weight_decay":
                    WEIGHT_DECAY,

                "aggregation":
                    "mean_frame_logits",

                "common_pool":
                    str(
                        COMMON_POOL_PATH
                    ),

            },

            BEST_CHECKPOINT

        )


    # -----------------------------------------------------------------
    # History
    # -----------------------------------------------------------------

    history.append(

        {

            "epoch":
                epoch,

            "train_loss":
                epoch_loss,

            "train_top1":
                epoch_top1,

            "train_top5":
                epoch_top5,

            "backbone_lr":
                backbone_current_lr,

            "classifier_lr":
                classifier_current_lr,

            "epoch_seconds":
                epoch_seconds,

            "best":
                is_best

        }

    )


    pd.DataFrame(
        history
    ).to_csv(
        HISTORY_PATH,
        index=False
    )


    # -----------------------------------------------------------------
    # Epoch report
    # -----------------------------------------------------------------

    print()
    print(
        f"Epoch {epoch:02d}/{EPOCHS} "
        f"COMPLETE"
    )

    print(
        f"  Loss       : "
        f"{epoch_loss:.4f}"
    )

    print(
        f"  Train Top1 : "
        f"{epoch_top1*100:.2f}%"
    )

    print(
        f"  Train Top5 : "
        f"{epoch_top5*100:.2f}%"
    )

    print(
        f"  Backbone LR : "
        f"{backbone_current_lr:.2e}"
    )

    print(
        f"  Classifier LR: "
        f"{classifier_current_lr:.2e}"
    )

    print(
        f"  Time       : "
        f"{epoch_seconds/60:.2f} min"
    )

    if is_best:

        print(
            "  *** NEW BEST ***"
        )


    # -----------------------------------------------------------------
    # IMPORTANT EARLY DIAGNOSTIC
    # -----------------------------------------------------------------

    if epoch == 3:

        print()
        print(
            "=" * 70
        )

        print(
            "EARLY CONVERGENCE CHECK"
        )

        print(
            "=" * 70
        )

        print(
            f"Epoch 3 Top-1: "
            f"{epoch_top1*100:.2f}%"
        )

        print(
            f"Epoch 3 Top-5: "
            f"{epoch_top5*100:.2f}%"
        )

        if epoch_top1 < 0.005:

            print()
            print(
                "WARNING:"
            )

            print(
                "ResNet18 remains extremely close to chance "
                "after 3 epochs."
            )

            print(
                "Do NOT simply increase model size."
            )

            print(
                "If this remains true, the next step should "
                "be a small overfit sanity test / label inspection."
            )


# =============================================================================
# 22. FINAL CHECKPOINT
# =============================================================================

total_training_seconds = (
    time.time()
    -
    training_start
)


torch.save(

    {

        "stage": "4.4",

        "version": "v2",

        "model": "ResNet18",

        "epoch": EPOCHS,

        "model_state_dict":
            model.state_dict(),

        "optimizer_state_dict":
            optimizer.state_dict(),

        "best_train_top1":
            best_top1,

        "best_train_top5":
            best_top5,

        "best_epoch":
            best_epoch,

        "final_train_top1":
            history[-1]["train_top1"],

        "final_train_top5":
            history[-1]["train_top5"],

        "final_train_loss":
            history[-1]["train_loss"],

        "seed":
            SEED,

        "num_classes":
            NUM_CLASSES,

        "image_size":
            IMAGE_SIZE,

        "frames_per_video":
            NUM_FRAMES,

        "batch_size":
            BATCH_SIZE,

        "backbone_lr":
            BACKBONE_LR,

        "classifier_lr":
            CLASSIFIER_LR,

        "weight_decay":
            WEIGHT_DECAY,

        "aggregation":
            "mean_frame_logits",

        "common_pool":
            str(
                COMMON_POOL_PATH
            )

    },

    FINAL_CHECKPOINT

)


# =============================================================================
# 23. FINAL SUMMARY
# =============================================================================

print()
print("=" * 90)
print("STAGE 4.4 RESNET18 v2 COMPLETE")
print("=" * 90)

print()
print(
    f"Readable videos       : "
    f"{len(common_pool):,}"
)

print(
    f"Classes represented   : "
    f"{common_pool['class_id'].nunique():,}"
)

print(
    f"Frames/video          : "
    f"{NUM_FRAMES}"
)

print()
print(
    f"Best epoch            : "
    f"{best_epoch}"
)

print(
    f"Best training Top-1   : "
    f"{best_top1*100:.2f}%"
)

print(
    f"Best training Top-5   : "
    f"{best_top5*100:.2f}%"
)

print(
    f"Final training Top-1  : "
    f"{history[-1]['train_top1']*100:.2f}%"
)

print(
    f"Final training Top-5  : "
    f"{history[-1]['train_top5']*100:.2f}%"
)

print(
    f"Total training time   : "
    f"{total_training_seconds/60:.2f} min"
)

print()
print("Artifacts:")

print(
    f"  Best checkpoint:\n"
    f"    {BEST_CHECKPOINT}"
)

print(
    f"  Final checkpoint:\n"
    f"    {FINAL_CHECKPOINT}"
)

print(
    f"  Training history:\n"
    f"    {HISTORY_PATH}"
)

print(
    f"  Configuration:\n"
    f"    {CONFIG_PATH}"
)

print()
print("=" * 90)
print("SCIENTIFIC NOTE")
print("=" * 90)

print(
    "This experiment uses the existing frozen Stage 4.4 "
    "common readable UID pool and frame cache."
)

print(
    "The 8 sampled frames are aggregated at video level "
    "before the classification loss."
)

print(
    "Training accuracy is a convergence diagnostic only."
)

print(
    "Official CISLR recognition evaluation remains "
    "prototype-to-test retrieval."
)

print(
    "The same frozen common pool must be reused for "
    "MobileNetV3-Large and EfficientNet-B0."
)

print("=" * 90)

CISLR — STAGE 4.4 RESNET18 v2
Device              : cuda
GPU                 : NVIDIA GeForce RTX 4050 Laptop GPU
GPU VRAM            : 5.64 GB
Classes             : 4,764
Frames/video        : 8
Image size          : 224x224
Batch size          : 16
Epochs              : 25
Backbone LR         : 1e-05
Classifier LR       : 0.0005
Weight decay        : 0.0001
Warmup epochs       : 1
Gradient clipping   : 1.0
AMP                 : True

Existing Stage 4.4 cache will be reused.

LOADING FROZEN COMMON READABLE POOL
Readable videos     : 4,417
Unique classes      : 4,417

VERIFYING COMMON POOL
Pool integrity checks PASSED.

VERIFYING CACHED FRAME FORMAT
Verified 100 representative cache files.
Cache format PASSED.

TRAINING DATA READY
Videos              : 4,417
Frames/video        : 8
Total frame samples : 35,336
Batches/epoch       : 277

BUILDING RESNET18
Parameters          : 13,620,444
Trainable parameters: 13,620,444

STARTING RESNET18 v2 TRAINING

Epoch 1:
  Classifier warm-up
Epoch

# CISLR — STAGE 4.4 MobileNetV3-Large v2

In [34]:
# =============================================================================
# CISLR — STAGE 4.4 MOBILENETV3-LARGE v2
#
# IMPROVED CONVERGENCE / TRAINING VERSION
#
# IMPORTANT:
# - Stage 4.1 READ ONLY
# - Stage 4.2 READ ONLY
# - Stage 4.3 READ ONLY
# - Existing Stage 4.4 common pool READ ONLY
# - Existing frame cache READ ONLY
# - NO VIDEO RE-DECODING
# - NO CACHE REBUILDING
# - SAME FROZEN POOL AS RESNET18 v2
#
# Purpose:
# Compare MobileNetV3-Large against ResNet18 under the same
# video-level training protocol.
#
# Official CISLR recognition evaluation remains:
# prototype -> test retrieval
#
# This experiment is a TRAINING / MODEL-COMPARISON DIAGNOSTIC.
# =============================================================================


import os
import gc
import json
import time
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

import torch
import torch.nn as nn

from torch.utils.data import Dataset, DataLoader

from torchvision import models, transforms

from PIL import Image

warnings.filterwarnings("ignore")


# =============================================================================
# 1. EXACT EXISTING STAGE 4.4 PATHS
# =============================================================================

PROJECT_ROOT = Path(
    "/home/dipshikha/Music/cao"
)

STAGE44_DIR = (
    PROJECT_ROOT
    / "CISLR_RESEARCH"
    / "audit"
    / "stage4_4_training"
)

MODEL_DIR = (
    PROJECT_ROOT
    / "CISLR_RESEARCH"
    / "models"
    / "stage4_4"
)

COMMON_POOL_PATH = (
    STAGE44_DIR
    / "stage4_4_common_readable_uid_pool.csv"
)

CACHE_DIR = (
    STAGE44_DIR
    / "frame_cache"
)


# =============================================================================
# 2. PATH VALIDATION
# =============================================================================

print("=" * 90)
print("CISLR — STAGE 4.4 MOBILENETV3-LARGE v2")
print("=" * 90)

print()
print("Checking existing Stage 4.4 artifacts...")

if not COMMON_POOL_PATH.exists():

    raise FileNotFoundError(
        f"""
Existing Stage 4.4 common pool not found:

{COMMON_POOL_PATH}

DO NOT rebuild it automatically.
"""
    )

if not CACHE_DIR.exists():

    raise FileNotFoundError(
        f"""
Existing Stage 4.4 frame cache not found:

{CACHE_DIR}

DO NOT rebuild it automatically.
"""
    )

MODEL_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# =============================================================================
# 3. CONFIGURATION
# =============================================================================

SEED = 42

IMAGE_SIZE = 224

NUM_FRAMES = 8

# Keep identical to ResNet18 v2 for direct comparison.
NUM_CLASSES = 4764

EPOCHS = 25

# MobileNetV3-Large comfortably fits with this configuration.
BATCH_SIZE = 16

NUM_WORKERS = 2

USE_AMP = torch.cuda.is_available()

DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)


# -----------------------------------------------------------------------------
# Optimization
# -----------------------------------------------------------------------------

BACKBONE_LR = 1e-5

CLASSIFIER_LR = 5e-4

WEIGHT_DECAY = 1e-4

WARMUP_EPOCHS = 1

GRAD_CLIP = 1.0


# =============================================================================
# 4. REPRODUCIBILITY
# =============================================================================

def seed_everything(seed):

    random.seed(seed)

    np.random.seed(seed)

    torch.manual_seed(seed)

    if torch.cuda.is_available():

        torch.cuda.manual_seed(seed)

        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True

    torch.backends.cudnn.benchmark = False


seed_everything(SEED)


# =============================================================================
# 5. CLEAN CUDA MEMORY
# =============================================================================

gc.collect()

if torch.cuda.is_available():

    torch.cuda.empty_cache()


# =============================================================================
# 6. HEADER
# =============================================================================

print()

print(f"Device              : {DEVICE}")

if torch.cuda.is_available():

    print(
        f"GPU                 : "
        f"{torch.cuda.get_device_name(0)}"
    )

    total_vram = (
        torch.cuda.get_device_properties(0).total_memory
        / (1024 ** 3)
    )

    print(
        f"GPU VRAM            : "
        f"{total_vram:.2f} GB"
    )

print(
    f"Classifier outputs  : {NUM_CLASSES:,}"
)

print(
    f"Frames/video        : {NUM_FRAMES}"
)

print(
    f"Image size          : {IMAGE_SIZE}x{IMAGE_SIZE}"
)

print(
    f"Batch size          : {BATCH_SIZE}"
)

print(
    f"Epochs              : {EPOCHS}"
)

print(
    f"Backbone LR         : {BACKBONE_LR}"
)

print(
    f"Classifier LR       : {CLASSIFIER_LR}"
)

print(
    f"Weight decay        : {WEIGHT_DECAY}"
)

print(
    f"Warmup epochs       : {WARMUP_EPOCHS}"
)

print(
    f"Gradient clipping   : {GRAD_CLIP}"
)

print(
    f"AMP                 : {USE_AMP}"
)

print()

print("EXISTING CACHE WILL BE REUSED.")

print("NO VIDEO RE-DECODING WILL OCCUR.")

print()


# =============================================================================
# 7. LOAD FROZEN COMMON POOL
# =============================================================================

print("=" * 90)
print("LOADING FROZEN COMMON READABLE POOL")
print("=" * 90)

common_pool = pd.read_csv(
    COMMON_POOL_PATH
)

required_pool_columns = [
    "uid",
    "gloss",
    "class_id",
    "video_path",
    "frame_indices",
    "cache_path",
    "status",
]

missing_pool_columns = [
    c
    for c in required_pool_columns
    if c not in common_pool.columns
]

if missing_pool_columns:

    raise RuntimeError(
        "Common pool missing required columns:\n"
        + "\n".join(missing_pool_columns)
    )


# Keep ONLY the exact readable pool used by ResNet18 v2.
common_pool = (
    common_pool[
        common_pool["status"] == "OK"
    ]
    .copy()
    .reset_index(drop=True)
)


print(
    f"Readable videos     : "
    f"{len(common_pool):,}"
)

print(
    f"Unique classes      : "
    f"{common_pool['class_id'].nunique():,}"
)


# =============================================================================
# 8. EXACT POOL RECONCILIATION
# =============================================================================

print()

print("=" * 90)
print("VERIFYING FROZEN COMMON POOL")
print("=" * 90)


# Expected exact Stage 4.4 v2 pool.
EXPECTED_VIDEOS = 4417

if len(common_pool) != EXPECTED_VIDEOS:

    raise RuntimeError(
        f"""
Unexpected number of readable videos.

Expected : {EXPECTED_VIDEOS:,}
Found    : {len(common_pool):,}

The MobileNet experiment must use exactly the same
frozen pool as ResNet18 v2.
"""
    )


# -----------------------------------------------------------------------------
# UID uniqueness
# -----------------------------------------------------------------------------

if common_pool["uid"].duplicated().any():

    duplicates = common_pool[
        common_pool["uid"].duplicated(
            keep=False
        )
    ]

    raise RuntimeError(
        "Duplicate UID found:\n"
        + str(duplicates.head(20))
    )


# -----------------------------------------------------------------------------
# One class per UID
# -----------------------------------------------------------------------------

uid_class_counts = (
    common_pool
    .groupby("uid")["class_id"]
    .nunique()
)

if (uid_class_counts > 1).any():

    raise RuntimeError(
        "Some UID maps to multiple class IDs."
    )


# -----------------------------------------------------------------------------
# Cache path uniqueness
# -----------------------------------------------------------------------------

if common_pool["cache_path"].duplicated().any():

    duplicates = common_pool[
        common_pool["cache_path"].duplicated(
            keep=False
        )
    ]

    raise RuntimeError(
        "Duplicate cache_path detected."
    )


# -----------------------------------------------------------------------------
# Class ID range
# -----------------------------------------------------------------------------

if (
    common_pool["class_id"].min() < 0
    or
    common_pool["class_id"].max() >= NUM_CLASSES
):

    raise RuntimeError(
        "Class IDs outside expected 0..4763 range."
    )


# -----------------------------------------------------------------------------
# Cache existence
#
# IMPORTANT:
# Use the EXACT cache_path recorded in the frozen pool.
# Do NOT reconstruct cache filenames.
# -----------------------------------------------------------------------------

missing_cache = []

for cache_path in common_pool["cache_path"]:

    if not Path(cache_path).exists():

        missing_cache.append(
            cache_path
        )

        if len(missing_cache) >= 20:
            break


if missing_cache:

    raise RuntimeError(
        "Missing cached frame files:\n"
        + "\n".join(missing_cache)
    )


print()

print("Pool integrity checks PASSED.")

print(
    f"Verified videos     : "
    f"{len(common_pool):,}"
)

print(
    f"Verified cache files: "
    f"{common_pool['cache_path'].nunique():,}"
)

print(
    "Missing cache files : 0"
)


# =============================================================================
# 9. VERIFY CACHE CONTENT
# =============================================================================

print()

print("=" * 90)
print("VERIFYING CACHED FRAME FORMAT")
print("=" * 90)


def validate_cache_file(path):

    try:

        with np.load(
            path,
            allow_pickle=False
        ) as data:

            if "frames" not in data:

                return False

            frames = data["frames"]

            expected_shape = (
                NUM_FRAMES,
                IMAGE_SIZE,
                IMAGE_SIZE,
                3
            )

            if frames.shape != expected_shape:

                return False

            if frames.dtype != np.uint8:

                return False

        return True

    except Exception:

        return False


# Same representative verification strategy as ResNet18 v2.
cache_check_count = min(
    100,
    len(common_pool)
)

cache_samples = common_pool.iloc[
    np.linspace(
        0,
        len(common_pool) - 1,
        cache_check_count
    ).astype(int)
]

bad_cache = []

for _, row in cache_samples.iterrows():

    if not validate_cache_file(
        row["cache_path"]
    ):

        bad_cache.append(
            row["uid"]
        )


if bad_cache:

    raise RuntimeError(
        "Invalid cache files found:\n"
        + "\n".join(
            map(
                str,
                bad_cache[:20]
            )
        )
    )


print(
    f"Verified {cache_check_count} "
    "representative cache files."
)

print(
    "Cache format PASSED."
)

print(
    "No video files were opened."
)


# =============================================================================
# 10. DATASET
# =============================================================================

class CachedCISLRDataset(
    Dataset
):

    def __init__(
        self,
        dataframe,
        transform=None
    ):

        self.df = (
            dataframe
            .reset_index(drop=True)
        )

        self.transform = transform


    def __len__(self):

        return len(self.df)


    def __getitem__(
        self,
        index
    ):

        row = self.df.iloc[index]

        cache_path = row["cache_path"]

        try:

            # IMPORTANT:
            # Load the exact cache file referenced by the frozen pool.
            with np.load(
                cache_path,
                allow_pickle=False
            ) as data:

                frames = data["frames"]

        except Exception as e:

            raise RuntimeError(
                f"""
Could not load cached frames.

UID:
{row['uid']}

Cache:
{cache_path}

Error:
{e}
"""
            )


        expected_shape = (
            NUM_FRAMES,
            IMAGE_SIZE,
            IMAGE_SIZE,
            3
        )

        if frames.shape != expected_shape:

            raise RuntimeError(
                f"""
Unexpected cache shape.

UID:
{row['uid']}

Shape:
{frames.shape}

Expected:
{expected_shape}
"""
            )


        transformed_frames = []


        for frame in frames:

            image = Image.fromarray(
                frame
            )

            if self.transform is not None:

                image = self.transform(
                    image
                )

            transformed_frames.append(
                image
            )


        # [8, 3, 224, 224]
        frames_tensor = torch.stack(
            transformed_frames,
            dim=0
        )


        label = torch.tensor(
            int(row["class_id"]),
            dtype=torch.long
        )


        return (
            frames_tensor,
            label
        )


# =============================================================================
# 11. TRAINING TRANSFORM
#
# IDENTICAL TO RESNET18 v2
#
# - Resize
# - Small RandomAffine
# - NO horizontal flip
# =============================================================================

train_transform = transforms.Compose([

    transforms.Resize(
        (
            IMAGE_SIZE,
            IMAGE_SIZE
        )
    ),

    transforms.RandomAffine(
        degrees=5,

        translate=(
            0.05,
            0.05
        ),

        scale=(
            0.95,
            1.05
        )
    ),

    transforms.ToTensor(),

    transforms.Normalize(

        mean=[
            0.485,
            0.456,
            0.406
        ],

        std=[
            0.229,
            0.224,
            0.225
        ]
    )
])


# =============================================================================
# 12. DATALOADER
# =============================================================================

train_dataset = CachedCISLRDataset(
    common_pool,
    transform=train_transform
)


train_loader = DataLoader(

    train_dataset,

    batch_size=BATCH_SIZE,

    shuffle=True,

    num_workers=NUM_WORKERS,

    pin_memory=torch.cuda.is_available(),

    persistent_workers=(
        NUM_WORKERS > 0
    ),

    drop_last=False
)


print()

print("=" * 90)
print("TRAINING DATA READY")
print("=" * 90)

print(
    f"Videos              : "
    f"{len(train_dataset):,}"
)

print(
    f"Frames/video        : "
    f"{NUM_FRAMES}"
)

print(
    f"Total frame samples : "
    f"{len(train_dataset) * NUM_FRAMES:,}"
)

print(
    f"Batches/epoch       : "
    f"{len(train_loader):,}"
)


# =============================================================================
# 13. BUILD PRETRAINED MOBILENETV3-LARGE
# =============================================================================

print()

print("=" * 90)
print("BUILDING MOBILENETV3-LARGE")
print("=" * 90)


weights = (
    models.MobileNet_V3_Large_Weights.DEFAULT
)


model = models.mobilenet_v3_large(
    weights=weights
)


# -----------------------------------------------------------------------------
# Replace ImageNet classifier
#
# torchvision MobileNetV3-Large:
# model.classifier[-1] = final Linear layer
# -----------------------------------------------------------------------------

classifier_in_features = (
    model.classifier[-1].in_features
)


model.classifier[-1] = nn.Linear(
    classifier_in_features,
    NUM_CLASSES
)


model = model.to(
    DEVICE
)


total_params = sum(
    p.numel()
    for p in model.parameters()
)

trainable_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)


print(
    f"Parameters          : "
    f"{total_params:,}"
)

print(
    f"Trainable parameters: "
    f"{trainable_params:,}"
)

print(
    f"Classifier input dim: "
    f"{classifier_in_features}"
)


# =============================================================================
# 14. LOSS
# =============================================================================

criterion = nn.CrossEntropyLoss()


# =============================================================================
# 15. DISCRIMINATIVE OPTIMIZER
#
# Backbone:
#     1e-5
#
# Classifier:
#     5e-4
# =============================================================================

backbone_parameters = []

classifier_parameters = []


for name, parameter in model.named_parameters():

    if name.startswith(
        "classifier."
    ):

        classifier_parameters.append(
            parameter
        )

    else:

        backbone_parameters.append(
            parameter
        )


optimizer = torch.optim.AdamW(

    [

        {
            "params": backbone_parameters,
            "lr": BACKBONE_LR
        },

        {
            "params": classifier_parameters,
            "lr": CLASSIFIER_LR
        }

    ],

    weight_decay=WEIGHT_DECAY
)


# =============================================================================
# 16. COSINE LR SCHEDULER
# =============================================================================

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(

    optimizer,

    T_max=max(
        1,
        EPOCHS - WARMUP_EPOCHS
    ),

    eta_min=1e-7
)


# =============================================================================
# 17. AMP
# =============================================================================

if USE_AMP:

    scaler = torch.cuda.amp.GradScaler()

else:

    scaler = None


# =============================================================================
# 18. CLASSIFIER WARM-UP CONTROL
# =============================================================================

def set_backbone_trainable(
    model,
    trainable
):

    for name, parameter in model.named_parameters():

        if name.startswith(
            "classifier."
        ):

            parameter.requires_grad = True

        else:

            parameter.requires_grad = trainable


# Epoch 1:
# classifier only

set_backbone_trainable(
    model,
    trainable=False
)


# =============================================================================
# 19. ARTIFACT DIRECTORIES
# =============================================================================

MOBILENET_AUDIT_DIR = (
    STAGE44_DIR
    / "mobilenet_v3_large_v2"
)

MOBILENET_MODEL_DIR = (
    MODEL_DIR
    / "mobilenet_v3_large_v2"
)


MOBILENET_AUDIT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

MOBILENET_MODEL_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# =============================================================================
# 20. ARTIFACT PATHS
# =============================================================================

BEST_CHECKPOINT = (
    MOBILENET_MODEL_DIR
    / "mobilenet_v3_large_stage4_4_v2_best.pt"
)

FINAL_CHECKPOINT = (
    MOBILENET_MODEL_DIR
    / "mobilenet_v3_large_stage4_4_v2_final.pt"
)

HISTORY_PATH = (
    MOBILENET_AUDIT_DIR
    / "mobilenet_v3_large_stage4_4_v2_training_history.csv"
)

CONFIG_PATH = (
    MOBILENET_AUDIT_DIR
    / "mobilenet_v3_large_stage4_4_v2_config.json"
)


# =============================================================================
# 21. CONFIGURATION RECORD
# =============================================================================

config = {

    "stage": "4.4",

    "version": "v2",

    "model": "MobileNetV3-Large",

    "dataset": "CISLR v1.5-a",

    "seed": SEED,

    "num_classes": NUM_CLASSES,

    "readable_videos": len(common_pool),

    "represented_classes":
        int(
            common_pool["class_id"].nunique()
        ),

    "image_size": IMAGE_SIZE,

    "frames_per_video": NUM_FRAMES,

    "epochs": EPOCHS,

    "batch_size": BATCH_SIZE,

    "backbone_lr": BACKBONE_LR,

    "classifier_lr": CLASSIFIER_LR,

    "weight_decay": WEIGHT_DECAY,

    "warmup_epochs": WARMUP_EPOCHS,

    "gradient_clip": GRAD_CLIP,

    "amp": USE_AMP,

    "aggregation":
        "mean_frame_logits",

    "horizontal_flip":
        False,

    "augmentation": {

        "Resize": [
            IMAGE_SIZE,
            IMAGE_SIZE
        ],

        "RandomAffine": {

            "degrees": 5,

            "translate": [
                0.05,
                0.05
            ],

            "scale": [
                0.95,
                1.05
            ]
        }
    },

    "common_pool":
        str(COMMON_POOL_PATH),

    "frame_cache":
        str(CACHE_DIR),

    "cache_source":
        "cache_path_column_in_frozen_common_pool",

    "best_checkpoint":
        str(BEST_CHECKPOINT),

    "final_checkpoint":
        str(FINAL_CHECKPOINT),

    "comparison_protocol":
        "same_as_resnet18_stage4_4_v2"

}


with open(
    CONFIG_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        config,
        f,
        indent=2
    )


# =============================================================================
# 22. TOP-K FUNCTION
# =============================================================================

def calculate_topk(
    logits,
    labels,
    k=5
):

    max_k = min(
        k,
        logits.size(1)
    )

    _, predictions = (
        logits.topk(
            max_k,
            dim=1
        )
    )

    correct = (
        predictions
        .eq(
            labels.unsqueeze(1)
        )
    )

    top1 = (
        correct[:, :1]
        .any(dim=1)
        .float()
        .sum()
        .item()
    )

    top5 = (
        correct[:, :max_k]
        .any(dim=1)
        .float()
        .sum()
        .item()
    )

    return top1, top5


# =============================================================================
# 23. TRAINING
# =============================================================================

print()

print("=" * 90)
print("STARTING MOBILENETV3-LARGE v2 TRAINING")
print("=" * 90)

print()

print("Epoch 1:")

print(
    "  Classifier warm-up"
)

print()

print("Epoch 2+:")

print(
    "  Full-network fine-tuning"
)

print()

print("Video-level prediction:")

print(
    "  8 frame logits -> mean -> class prediction"
)

print()

print("Cache:")

print(
    "  Existing Stage 4.4 cache"
)

print()

print("MP4 decoding:")

print(
    "  DISABLED"
)

print()


history = []

best_top1 = -1.0

best_top5 = -1.0

best_epoch = -1

training_start = time.time()


for epoch in range(
    1,
    EPOCHS + 1
):


    # -------------------------------------------------------------------------
    # Unfreeze after warm-up
    # -------------------------------------------------------------------------

    if epoch == WARMUP_EPOCHS + 1:

        set_backbone_trainable(
            model,
            trainable=True
        )

        print()

        print(
            "Backbone UNFROZEN — "
            "full-network fine-tuning begins."
        )


    model.train()


    epoch_start = time.time()


    running_loss = 0.0

    correct_top1 = 0.0

    correct_top5 = 0.0

    total_videos = 0


    num_batches = len(
        train_loader
    )


    print()

    print(
        f"-- Epoch {epoch:02d}/{EPOCHS} starting --"
    )


    for batch_idx, (
        frames,
        labels
    ) in enumerate(
        train_loader,
        start=1
    ):


        # ---------------------------------------------------------------------
        # frames:
        #
        # [B, 8, 3, 224, 224]
        # ---------------------------------------------------------------------

        frames = frames.to(
            DEVICE,
            non_blocking=True
        )

        labels = labels.to(
            DEVICE,
            non_blocking=True
        )


        B = frames.shape[0]


        # ---------------------------------------------------------------------
        # Flatten frames
        #
        # [B, 8, 3, 224, 224]
        #
        # ->
        #
        # [B*8, 3, 224, 224]
        # ---------------------------------------------------------------------

        frame_input = frames.reshape(
            B * NUM_FRAMES,
            3,
            IMAGE_SIZE,
            IMAGE_SIZE
        )


        optimizer.zero_grad(
            set_to_none=True
        )


        # ---------------------------------------------------------------------
        # Forward + video-level aggregation
        # ---------------------------------------------------------------------

        if USE_AMP:

            with torch.cuda.amp.autocast():

                frame_logits = model(
                    frame_input
                )

                # [B*8, C]
                #
                # ->
                #
                # [B, 8, C]

                frame_logits = (
                    frame_logits
                    .reshape(
                        B,
                        NUM_FRAMES,
                        NUM_CLASSES
                    )
                )


                # -------------------------------------------------------------
                # Video-level mean aggregation
                # -------------------------------------------------------------

                video_logits = (
                    frame_logits
                    .mean(dim=1)
                )


                # -------------------------------------------------------------
                # Classification loss
                # -------------------------------------------------------------

                loss = criterion(
                    video_logits,
                    labels
                )


            scaler.scale(
                loss
            ).backward()


            scaler.unscale_(
                optimizer
            )


            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                GRAD_CLIP
            )


            scaler.step(
                optimizer
            )


            scaler.update()


        else:

            frame_logits = model(
                frame_input
            )

            frame_logits = (
                frame_logits
                .reshape(
                    B,
                    NUM_FRAMES,
                    NUM_CLASSES
                )
            )

            video_logits = (
                frame_logits
                .mean(dim=1)
            )

            loss = criterion(
                video_logits,
                labels
            )

            loss.backward()

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                GRAD_CLIP
            )

            optimizer.step()


        # ---------------------------------------------------------------------
        # Metrics
        # ---------------------------------------------------------------------

        top1_batch, top5_batch = (
            calculate_topk(
                video_logits,
                labels,
                k=5
            )
        )


        correct_top1 += top1_batch

        correct_top5 += top5_batch

        total_videos += B


        running_loss += (
            loss.item() * B
        )


        # ---------------------------------------------------------------------
        # Progress
        # ---------------------------------------------------------------------

        progress_interval = max(
            1,
            num_batches // 20
        )


        if (
            batch_idx == 1
            or
            batch_idx % progress_interval == 0
            or
            batch_idx == num_batches
        ):

            elapsed = (
                time.time()
                - epoch_start
            )

            rate = (
                batch_idx
                /
                max(
                    elapsed,
                    1e-6
                )
            )

            remaining = (
                num_batches
                - batch_idx
            )

            eta_seconds = (
                remaining
                /
                max(
                    rate,
                    1e-6
                )
            )

            current_loss = (
                running_loss
                /
                max(
                    total_videos,
                    1
                )
            )

            current_top1 = (
                correct_top1
                /
                max(
                    total_videos,
                    1
                )
            )

            current_top5 = (
                correct_top5
                /
                max(
                    total_videos,
                    1
                )
            )


            print(
                f"  batch "
                f"{batch_idx:,}/"
                f"{num_batches:,}"
                f" | loss "
                f"{current_loss:.4f}"
                f" | Top1 "
                f"{current_top1 * 100:.2f}%"
                f" | Top5 "
                f"{current_top5 * 100:.2f}%"
                f" | ETA "
                f"{eta_seconds / 60:.1f} min"
            )


    # =========================================================================
    # END EPOCH
    # =========================================================================

    epoch_seconds = (
        time.time()
        - epoch_start
    )


    epoch_loss = (
        running_loss
        /
        max(
            total_videos,
            1
        )
    )


    epoch_top1 = (
        correct_top1
        /
        max(
            total_videos,
            1
        )
    )


    epoch_top5 = (
        correct_top5
        /
        max(
            total_videos,
            1
        )
    )


    # -------------------------------------------------------------------------
    # Scheduler
    #
    # Keep LR constant during epoch 1.
    # Start cosine schedule after warm-up.
    # -------------------------------------------------------------------------

    if epoch > WARMUP_EPOCHS:

        scheduler.step()


    # -------------------------------------------------------------------------
    # Current learning rates
    # -------------------------------------------------------------------------

    backbone_current_lr = (
        optimizer.param_groups[0]["lr"]
    )

    classifier_current_lr = (
        optimizer.param_groups[1]["lr"]
    )


    # -------------------------------------------------------------------------
    # Best model
    # -------------------------------------------------------------------------

    is_best = (
        epoch_top1 > best_top1
    )


    if is_best:

        best_top1 = epoch_top1

        best_top5 = epoch_top5

        best_epoch = epoch


        torch.save(

            {

                "stage":
                    "4.4",

                "version":
                    "v2",

                "model":
                    "MobileNetV3-Large",

                "epoch":
                    epoch,

                "model_state_dict":
                    model.state_dict(),

                "optimizer_state_dict":
                    optimizer.state_dict(),

                "train_loss":
                    epoch_loss,

                "train_top1":
                    epoch_top1,

                "train_top5":
                    epoch_top5,

                "best_train_top1":
                    best_top1,

                "best_train_top5":
                    best_top5,

                "best_epoch":
                    best_epoch,

                "seed":
                    SEED,

                "num_classes":
                    NUM_CLASSES,

                "image_size":
                    IMAGE_SIZE,

                "frames_per_video":
                    NUM_FRAMES,

                "batch_size":
                    BATCH_SIZE,

                "backbone_lr":
                    BACKBONE_LR,

                "classifier_lr":
                    CLASSIFIER_LR,

                "weight_decay":
                    WEIGHT_DECAY,

                "aggregation":
                    "mean_frame_logits",

                "common_pool":
                    str(COMMON_POOL_PATH),

                "frame_cache":
                    str(CACHE_DIR)

            },

            BEST_CHECKPOINT
        )


    # -------------------------------------------------------------------------
    # History
    # -------------------------------------------------------------------------

    history.append(

        {

            "epoch":
                epoch,

            "train_loss":
                epoch_loss,

            "train_top1":
                epoch_top1,

            "train_top5":
                epoch_top5,

            "backbone_lr":
                backbone_current_lr,

            "classifier_lr":
                classifier_current_lr,

            "epoch_seconds":
                epoch_seconds,

            "best":
                is_best

        }

    )


    pd.DataFrame(
        history
    ).to_csv(
        HISTORY_PATH,
        index=False
    )


    # -------------------------------------------------------------------------
    # Epoch report
    # -------------------------------------------------------------------------

    print()

    print(
        f"Epoch {epoch:02d}/{EPOCHS} COMPLETE"
    )

    print(
        f"  Loss        : "
        f"{epoch_loss:.4f}"
    )

    print(
        f"  Train Top1  : "
        f"{epoch_top1 * 100:.2f}%"
    )

    print(
        f"  Train Top5  : "
        f"{epoch_top5 * 100:.2f}%"
    )

    print(
        f"  Backbone LR  : "
        f"{backbone_current_lr:.2e}"
    )

    print(
        f"  Classifier LR: "
        f"{classifier_current_lr:.2e}"
    )

    print(
        f"  Time         : "
        f"{epoch_seconds / 60:.2f} min"
    )


    if is_best:

        print(
            "  *** NEW BEST ***"
        )


    # -------------------------------------------------------------------------
    # Early convergence diagnostic
    # -------------------------------------------------------------------------

    if epoch == 3:

        print()

        print("=" * 70)

        print(
            "EARLY CONVERGENCE CHECK"
        )

        print("=" * 70)

        print(
            f"Epoch 3 Top-1: "
            f"{epoch_top1 * 100:.2f}%"
        )

        print(
            f"Epoch 3 Top-5: "
            f"{epoch_top5 * 100:.2f}%"
        )

        if epoch_top1 < 0.005:

            print()

            print(
                "WARNING:"
            )

            print(
                "MobileNetV3-Large remains extremely "
                "close to chance after 3 epochs."
            )

            print(
                "Do NOT automatically increase model size."
            )

            print(
                "Continue the experiment and compare "
                "the final convergence behavior."
            )


# =============================================================================
# 24. FINAL CHECKPOINT
# =============================================================================

total_training_seconds = (
    time.time()
    - training_start
)


torch.save(

    {

        "stage":
            "4.4",

        "version":
            "v2",

        "model":
            "MobileNetV3-Large",

        "epoch":
            EPOCHS,

        "model_state_dict":
            model.state_dict(),

        "optimizer_state_dict":
            optimizer.state_dict(),

        "best_train_top1":
            best_top1,

        "best_train_top5":
            best_top5,

        "best_epoch":
            best_epoch,

        "final_train_top1":
            history[-1]["train_top1"],

        "final_train_top5":
            history[-1]["train_top5"],

        "final_train_loss":
            history[-1]["train_loss"],

        "seed":
            SEED,

        "num_classes":
            NUM_CLASSES,

        "image_size":
            IMAGE_SIZE,

        "frames_per_video":
            NUM_FRAMES,

        "batch_size":
            BATCH_SIZE,

        "backbone_lr":
            BACKBONE_LR,

        "classifier_lr":
            CLASSIFIER_LR,

        "weight_decay":
            WEIGHT_DECAY,

        "aggregation":
            "mean_frame_logits",

        "common_pool":
            str(COMMON_POOL_PATH),

        "frame_cache":
            str(CACHE_DIR)

    },

    FINAL_CHECKPOINT
)


# =============================================================================
# 25. FINAL SUMMARY
# =============================================================================

print()

print("=" * 90)

print(
    "STAGE 4.4 MOBILENETV3-LARGE v2 COMPLETE"
)

print("=" * 90)

print()

print(
    f"Readable videos       : "
    f"{len(common_pool):,}"
)

print(
    f"Classes represented   : "
    f"{common_pool['class_id'].nunique():,}"
)

print(
    f"Classifier outputs    : "
    f"{NUM_CLASSES:,}"
)

print(
    f"Frames/video          : "
    f"{NUM_FRAMES}"
)

print()

print(
    f"Best epoch            : "
    f"{best_epoch}"
)

print(
    f"Best training Top-1   : "
    f"{best_top1 * 100:.2f}%"
)

print(
    f"Best training Top-5   : "
    f"{best_top5 * 100:.2f}%"
)

print(
    f"Final training Top-1  : "
    f"{history[-1]['train_top1'] * 100:.2f}%"
)

print(
    f"Final training Top-5  : "
    f"{history[-1]['train_top5'] * 100:.2f}%"
)

print(
    f"Total training time   : "
    f"{total_training_seconds / 60:.2f} min"
)

print()

print("Artifacts:")

print()

print(
    "  Best checkpoint:"
)

print(
    f"    {BEST_CHECKPOINT}"
)

print()

print(
    "  Final checkpoint:"
)

print(
    f"    {FINAL_CHECKPOINT}"
)

print()

print(
    "  Training history:"
)

print(
    f"    {HISTORY_PATH}"
)

print()

print(
    "  Configuration:"
)

print(
    f"    {CONFIG_PATH}"
)

print()

print("=" * 90)

print(
    "SCIENTIFIC NOTE"
)

print("=" * 90)

print()

print(
    "This experiment reuses the exact frozen Stage 4.4 "
    "common readable UID pool used by ResNet18 v2."
)

print()

print(
    "The exact cache_path recorded in the common pool "
    "is used for every video."
)

print()

print(
    "No MP4 files are decoded or reprocessed."
)

print()

print(
    "The 8 sampled frames are aggregated at video level "
    "before the classification loss."
)

print()

print(
    "Training accuracy is a convergence/model-comparison "
    "diagnostic only."
)

print()

print(
    "Official CISLR recognition evaluation remains "
    "prototype-to-test retrieval."
)

print()

print(
    "MobileNetV3-Large results will later be compared "
    "with ResNet18 and EfficientNet-B0 under the same "
    "Stage 4.4 protocol."
)

print()

print("=" * 90)

CISLR — STAGE 4.4 MOBILENETV3-LARGE v2

Checking existing Stage 4.4 artifacts...

Device              : cuda
GPU                 : NVIDIA GeForce RTX 4050 Laptop GPU
GPU VRAM            : 5.64 GB
Classifier outputs  : 4,764
Frames/video        : 8
Image size          : 224x224
Batch size          : 16
Epochs              : 25
Backbone LR         : 1e-05
Classifier LR       : 0.0005
Weight decay        : 0.0001
Warmup epochs       : 1
Gradient clipping   : 1.0
AMP                 : True

EXISTING CACHE WILL BE REUSED.
NO VIDEO RE-DECODING WILL OCCUR.

LOADING FROZEN COMMON READABLE POOL
Readable videos     : 4,417
Unique classes      : 4,417

VERIFYING FROZEN COMMON POOL

Pool integrity checks PASSED.
Verified videos     : 4,417
Verified cache files: 4,417
Missing cache files : 0

VERIFYING CACHED FRAME FORMAT
Verified 100 representative cache files.
Cache format PASSED.
No video files were opened.

TRAINING DATA READY
Videos              : 4,417
Frames/video        : 8
Total frame samp

STAGE 4.4
# EfficientNet-B0 v2 Training

In [1]:
# ============================================================
# CISLR — STAGE 4.4
# EfficientNet-B0 v2
# EXACT VIDEO-LEVEL MEAN-LOGIT OBJECTIVE
#
# Fresh training run.
#
# Objective:
#   8 frames -> EfficientNet logits
#   -> mean logits across 8 frames
#   -> ONE CE loss per video
#
# Memory strategy:
#   Video batch = 1
#   Frame micro-batch = 1
#   Gradient accumulation = 16 videos
#   Effective batch size = 16
#   Entire EfficientNet feature extractor checkpointed
#
# Cache format:
#   .npz
#   key = "frames"
#   shape = (8, 224, 224, 3)
#   dtype = uint8
# ============================================================


# ============================================================
# 1. CUDA allocator
# ============================================================

import os

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"


# ============================================================
# 2. Imports
# ============================================================

import gc
import json
import time
import random

import numpy as np
import pandas as pd

from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader

from torchvision import models
from torchvision.transforms import functional as TF
from torchvision.transforms import InterpolationMode

from torch.utils.checkpoint import checkpoint


# ============================================================
# 3. Reproducibility
# ============================================================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.benchmark = True

torch.set_float32_matmul_precision("high")


# ============================================================
# 4. Device
# ============================================================

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("=" * 80)
print("DEVICE")
print("=" * 80)

print("Device:", DEVICE)

if DEVICE.type == "cuda":

    print(
        "GPU:",
        torch.cuda.get_device_name(0)
    )

    print(
        "CUDA:",
        torch.version.cuda
    )

    free_mem, total_mem = (
        torch.cuda.mem_get_info()
    )

    print(
        f"Free GPU memory: "
        f"{free_mem / 1024**3:.2f} GB"
    )

    print(
        f"Total GPU memory: "
        f"{total_mem / 1024**3:.2f} GB"
    )


# ============================================================
# 5. Paths
# ============================================================

PROJECT_ROOT = Path(
    "/home/dipshikha/Music/cao"
)

RESEARCH_ROOT = (
    PROJECT_ROOT /
    "CISLR_RESEARCH"
)

COMMON_POOL_PATH = (
    RESEARCH_ROOT /
    "audit" /
    "stage4_4_training" /
    "stage4_4_common_readable_uid_pool.csv"
)

CACHE_ROOT = (
    RESEARCH_ROOT /
    "audit" /
    "stage4_4_training" /
    "frame_cache"
)


# ============================================================
# NEW EXPERIMENT DIRECTORY
# ============================================================

RUN_ROOT = (
    RESEARCH_ROOT /
    "models" /
    "stage4_4" /
    "efficientnet_b0_v2_exact_meanlogit"
)

AUDIT_ROOT = (
    RESEARCH_ROOT /
    "audit" /
    "stage4_4_training" /
    "efficientnet_b0_v2_exact_meanlogit"
)

RUN_ROOT.mkdir(
    parents=True,
    exist_ok=True
)

AUDIT_ROOT.mkdir(
    parents=True,
    exist_ok=True
)


BEST_MODEL_PATH = (
    RUN_ROOT /
    "efficientnet_b0_stage4_4_v2_exact_meanlogit_best.pt"
)

FINAL_MODEL_PATH = (
    RUN_ROOT /
    "efficientnet_b0_stage4_4_v2_exact_meanlogit_final.pt"
)

HISTORY_PATH = (
    AUDIT_ROOT /
    "efficientnet_b0_stage4_4_v2_exact_meanlogit_training_history.csv"
)

CONFIG_PATH = (
    AUDIT_ROOT /
    "efficientnet_b0_stage4_4_v2_exact_meanlogit_config.json"
)


# ============================================================
# 6. Configuration
# ============================================================

NUM_CLASSES = 4764

NUM_FRAMES = 8

IMAGE_SIZE = 224


# ------------------------------------------------------------
# Memory-safe training
# ------------------------------------------------------------

VIDEO_BATCH_SIZE = 1

FRAME_MICRO_BATCH = 1

ACCUM_STEPS = 16


# ------------------------------------------------------------
# Training
# ------------------------------------------------------------

NUM_EPOCHS = 25

WARMUP_EPOCHS = 1


# ------------------------------------------------------------
# Optimizer
# ------------------------------------------------------------

BACKBONE_LR = 1e-5

CLASSIFIER_LR = 5e-4

WEIGHT_DECAY = 1e-4

GRAD_CLIP = 1.0


# ------------------------------------------------------------
# DataLoader
#
# IMPORTANT:
# We use num_workers=0 because the cache is already local
# and this eliminates multiprocessing issues during training.
# ------------------------------------------------------------

NUM_WORKERS = 0

PIN_MEMORY = False


# ------------------------------------------------------------
# AMP
# ------------------------------------------------------------

USE_AMP = True


# ============================================================
# 7. ImageNet normalization
# ============================================================

IMAGENET_MEAN = torch.tensor(
    [0.485, 0.456, 0.406],
    dtype=torch.float32
)

IMAGENET_STD = torch.tensor(
    [0.229, 0.224, 0.225],
    dtype=torch.float32
)


# ============================================================
# 8. Load frozen Stage 4.4 pool
# ============================================================

print("\n" + "=" * 80)
print("LOADING FROZEN COMMON POOL")
print("=" * 80)

pool_df = pd.read_csv(
    COMMON_POOL_PATH
)

print(
    "Original pool shape:",
    pool_df.shape
)

print(
    "\nStatus counts:"
)

print(
    pool_df["status"].value_counts(
        dropna=False
    )
)


# ------------------------------------------------------------
# Keep only authoritative readable rows
# ------------------------------------------------------------

pool_df = pool_df[
    pool_df["status"]
    .astype(str)
    .str.upper()
    .eq("OK")
].copy()

pool_df = pool_df.reset_index(
    drop=True
)


# ------------------------------------------------------------
# Validate
# ------------------------------------------------------------

print(
    "\nReadable videos:",
    len(pool_df)
)

assert len(pool_df) == 4417

assert (
    pool_df["uid"].nunique()
    == 4417
)

assert (
    pool_df["class_id"].nunique()
    == 4417
)

assert (
    pool_df["class_id"].min()
    >= 0
)

assert (
    pool_df["class_id"].max()
    < NUM_CLASSES
)


# ============================================================
# 9. Verify cache files
# ============================================================

print("\n" + "=" * 80)
print("VERIFYING CACHE")
print("=" * 80)

missing_cache = []

for cache_path in pool_df["cache_path"]:

    p = Path(
        str(cache_path)
    )

    if not p.is_absolute():

        p = CACHE_ROOT / p

    if not p.exists():

        missing_cache.append(
            str(p)
        )


if missing_cache:

    print(
        "Missing cache files:",
        len(missing_cache)
    )

    for p in missing_cache[:10]:

        print(p)

    raise FileNotFoundError(
        "One or more cache files are missing."
    )


print(
    "All 4417 cache files exist."
)


# ============================================================
# 10. Dataset
# ============================================================

class CISLRVideoCacheDataset(
    Dataset
):


    def __init__(
        self,
        dataframe,
        num_frames=8,
        image_size=224,
        train=True
    ):

        self.df = dataframe.reset_index(
            drop=True
        )

        self.num_frames = num_frames

        self.image_size = image_size

        self.train = train

        self.mean = IMAGENET_MEAN.view(
            3,
            1,
            1
        )

        self.std = IMAGENET_STD.view(
            3,
            1,
            1
        )


    def __len__(self):

        return len(self.df)


    def _resolve_cache_path(
        self,
        value
    ):

        p = Path(
            str(value)
        )

        if p.is_absolute():

            return p

        return CACHE_ROOT / p


    def _load_frames(
        self,
        cache_path
    ):

        # ----------------------------------------------------
        # IMPORTANT FIX:
        #
        # np.load(.npz) returns NpzFile.
        # The actual array is stored under "frames".
        # ----------------------------------------------------

        npz = np.load(
            cache_path,
            allow_pickle=False
        )

        try:

            if "frames" not in npz.files:

                raise RuntimeError(
                    f"'frames' key missing in:\n"
                    f"{cache_path}\n"
                    f"Available keys: "
                    f"{npz.files}"
                )

            frames = npz["frames"]

        finally:

            npz.close()


        return frames


    def _transform_frame(
        self,
        frame
    ):

        # ----------------------------------------------------
        # uint8 HWC -> float CHW
        # ----------------------------------------------------

        x = torch.from_numpy(
            frame.copy()
        )

        x = x.permute(
            2,
            0,
            1
        )

        x = x.float() / 255.0


        # ----------------------------------------------------
        # Resize
        # ----------------------------------------------------

        x = TF.resize(
            x,
            [
                self.image_size,
                self.image_size
            ],
            interpolation=(
                InterpolationMode.BILINEAR
            ),
            antialias=True
        )


        # ----------------------------------------------------
        # Stage 4.4 augmentation
        #
        # RandomAffine:
        #   degrees = ±5
        #   translation = ±5%
        #   scale = 0.95–1.05
        #
        # NO horizontal flip.
        # ----------------------------------------------------

        if self.train:

            angle = random.uniform(
                -5.0,
                5.0
            )

            translate_x = int(
                random.uniform(
                    -0.05,
                    0.05
                )
                * self.image_size
            )

            translate_y = int(
                random.uniform(
                    -0.05,
                    0.05
                )
                * self.image_size
            )

            scale = random.uniform(
                0.95,
                1.05
            )

            x = TF.affine(
                x,
                angle=angle,
                translate=[
                    translate_x,
                    translate_y
                ],
                scale=scale,
                shear=[
                    0.0,
                    0.0
                ],
                interpolation=(
                    InterpolationMode.BILINEAR
                )
            )


        # ----------------------------------------------------
        # ImageNet normalization
        # ----------------------------------------------------

        x = (
            x - self.mean
        ) / self.std


        return x


    def __getitem__(
        self,
        index
    ):

        row = self.df.iloc[
            index
        ]


        cache_path = (
            self._resolve_cache_path(
                row["cache_path"]
            )
        )


        if not cache_path.exists():

            raise FileNotFoundError(
                f"Cache not found:\n"
                f"{cache_path}"
            )


        # ----------------------------------------------------
        # Correct .npz loading
        # ----------------------------------------------------

        frames = self._load_frames(
            cache_path
        )


        # ----------------------------------------------------
        # Validate
        # ----------------------------------------------------

        expected_shape = (
            self.num_frames,
            self.image_size,
            self.image_size,
            3
        )


        if frames.shape != expected_shape:

            raise RuntimeError(
                f"Invalid cache shape.\n"
                f"File: {cache_path}\n"
                f"Got: {frames.shape}\n"
                f"Expected: {expected_shape}"
            )


        if frames.dtype != np.uint8:

            raise RuntimeError(
                f"Invalid cache dtype.\n"
                f"File: {cache_path}\n"
                f"Got: {frames.dtype}\n"
                f"Expected: uint8"
            )


        # ----------------------------------------------------
        # Transform 8 frames
        # ----------------------------------------------------

        transformed_frames = []


        for frame in frames:

            transformed_frames.append(
                self._transform_frame(
                    frame
                )
            )


        # ----------------------------------------------------
        # [8, 3, 224, 224]
        # ----------------------------------------------------

        video = torch.stack(
            transformed_frames,
            dim=0
        )


        # ----------------------------------------------------
        # Video-level label
        # ----------------------------------------------------

        label = torch.tensor(
            int(row["class_id"]),
            dtype=torch.long
        )


        return video, label


# ============================================================
# 11. Create dataset
# ============================================================

dataset = CISLRVideoCacheDataset(
    pool_df,
    num_frames=NUM_FRAMES,
    image_size=IMAGE_SIZE,
    train=True
)


# ============================================================
# 12. DataLoader
# ============================================================

train_loader = DataLoader(
    dataset,
    batch_size=VIDEO_BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY
)


print("\n" + "=" * 80)
print("DATALOADER")
print("=" * 80)

print(
    "Dataset videos:",
    len(dataset)
)

print(
    "DataLoader batches:",
    len(train_loader)
)

print(
    "Video batch:",
    VIDEO_BATCH_SIZE
)

print(
    "Frames/video:",
    NUM_FRAMES
)

print(
    "Frame micro-batch:",
    FRAME_MICRO_BATCH
)

print(
    "Accumulation steps:",
    ACCUM_STEPS
)

print(
    "Effective batch:",
    VIDEO_BATCH_SIZE * ACCUM_STEPS
)


# ============================================================
# 13. Mandatory one-batch validation
# ============================================================

print("\n" + "=" * 80)
print("VALIDATING DATA PIPELINE")
print("=" * 80)


test_video, test_label = dataset[0]

print(
    "Direct dataset shape:",
    test_video.shape
)

print(
    "Direct label:",
    test_label.item()
)


assert test_video.shape == (
    8,
    3,
    224,
    224
)


test_batch_videos, test_batch_labels = next(
    iter(train_loader)
)


print(
    "DataLoader video shape:",
    test_batch_videos.shape
)

print(
    "DataLoader label shape:",
    test_batch_labels.shape
)


assert test_batch_videos.shape == (
    1,
    8,
    3,
    224,
    224
)


# ============================================================
# 14. Build EfficientNet-B0
# ============================================================

print("\n" + "=" * 80)
print("BUILDING EFFICIENTNET-B0")
print("=" * 80)


weights = (
    models.EfficientNet_B0_Weights
    .IMAGENET1K_V1
)


model = models.efficientnet_b0(
    weights=weights
)


# ------------------------------------------------------------
# Replace classifier
# ------------------------------------------------------------

classifier_in_features = (
    model.classifier[1].in_features
)


model.classifier[1] = nn.Linear(
    classifier_in_features,
    NUM_CLASSES
)


print(
    "Classifier input:",
    classifier_in_features
)

print(
    "Classifier output:",
    NUM_CLASSES
)


# ============================================================
# 15. Activation checkpointing
#
# Entire feature extractor is checkpointed as ONE block.
# This is more memory-efficient than keeping checkpoints
# between every EfficientNet feature block.
# ============================================================

class CheckpointedFeatures(
    nn.Module
):


    def __init__(
        self,
        features
    ):

        super().__init__()

        self.features = features


    def forward(
        self,
        x
    ):

        if (
            self.training
            and torch.is_grad_enabled()
            and any(
                p.requires_grad
                for p in self.features.parameters()
            )
        ):

            return checkpoint(
                self.features,
                x,
                use_reentrant=False
            )


        return self.features(x)


# Replace original feature extractor
model.features = CheckpointedFeatures(
    model.features
)


# ============================================================
# 16. Move model to GPU
# ============================================================

model = model.to(
    DEVICE
)


# ============================================================
# 17. Parameter count
# ============================================================

total_params = sum(
    p.numel()
    for p in model.parameters()
)

print(
    "\nTotal parameters:",
    f"{total_params:,}"
)


# ============================================================
# 18. Classifier warmup
# ============================================================

for p in model.features.parameters():

    p.requires_grad = False


for p in model.classifier.parameters():

    p.requires_grad = True


print(
    "Epoch 1:"
)

print(
    "Backbone: FROZEN"
)

print(
    "Classifier: TRAINABLE"
)


# ============================================================
# 19. Optimizer
# ============================================================

optimizer = torch.optim.AdamW(
    [
        {
            "params":
                model.features.parameters(),

            "lr":
                BACKBONE_LR
        },

        {
            "params":
                model.classifier.parameters(),

            "lr":
                CLASSIFIER_LR
        }
    ],

    weight_decay=WEIGHT_DECAY,

    foreach=False
)


# ============================================================
# 20. Cosine scheduler
# ============================================================

scheduler = (
    torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=NUM_EPOCHS,
        eta_min=1e-7
    )
)


# ============================================================
# 21. AMP
# ============================================================

scaler = torch.cuda.amp.GradScaler(
    enabled=(
        USE_AMP
        and DEVICE.type == "cuda"
    )
)


# ============================================================
# 22. Helper functions
# ============================================================

def set_backbone_trainable(
    model,
    trainable=True
):

    for p in model.features.parameters():

        p.requires_grad = trainable


def count_trainable_parameters(
    model
):

    return sum(
        p.numel()
        for p in model.parameters()
        if p.requires_grad
    )


def topk_correct(
    logits,
    labels,
    k=5
):

    max_k = min(
        k,
        logits.size(1)
    )

    _, predictions = logits.topk(
        max_k,
        dim=1,
        largest=True,
        sorted=True
    )

    correct = (
        predictions
        == labels.unsqueeze(1)
    )

    top1 = (
        correct[:, :1]
        .any(dim=1)
        .sum()
        .item()
    )

    top5 = (
        correct[:, :max_k]
        .any(dim=1)
        .sum()
        .item()
    )

    return top1, top5


# ============================================================
# 23. Configuration
# ============================================================

config = {

    "experiment":
        "CISLR EfficientNet-B0 Stage 4.4 "
        "Exact Mean-Logit Video Objective",

    "seed":
        SEED,

    "pool":
        str(COMMON_POOL_PATH),

    "cache_root":
        str(CACHE_ROOT),

    "videos":
        len(dataset),

    "classes":
        NUM_CLASSES,

    "frames_per_video":
        NUM_FRAMES,

    "image_size":
        IMAGE_SIZE,

    "model":
        "EfficientNet-B0",

    "pretrained":
        "ImageNet-1K",

    "objective":
        "CE(mean(frame_logits), video_label)",

    "video_batch_size":
        VIDEO_BATCH_SIZE,

    "frame_micro_batch":
        FRAME_MICRO_BATCH,

    "gradient_accumulation":
        ACCUM_STEPS,

    "effective_batch_size":
        VIDEO_BATCH_SIZE * ACCUM_STEPS,

    "epochs":
        NUM_EPOCHS,

    "warmup_epochs":
        WARMUP_EPOCHS,

    "backbone_lr":
        BACKBONE_LR,

    "classifier_lr":
        CLASSIFIER_LR,

    "weight_decay":
        WEIGHT_DECAY,

    "gradient_clip":
        GRAD_CLIP,

    "activation_checkpointing":
        True,

    "checkpoint_scope":
        "entire EfficientNet feature extractor",

    "horizontal_flip":
        False,

    "random_affine":
        True,

    "random_affine_degrees":
        5,

    "random_affine_translation":
        0.05,

    "random_affine_scale":
        [
            0.95,
            1.05
        ],

    "amp":
        USE_AMP,

    "num_workers":
        NUM_WORKERS
}


with open(
    CONFIG_PATH,
    "w"
) as f:

    json.dump(
        config,
        f,
        indent=2
    )


print(
    "\nConfiguration saved:"
)

print(
    CONFIG_PATH
)


# ============================================================
# 24. FREE validation tensors
# ============================================================

del test_video
del test_label
del test_batch_videos
del test_batch_labels

gc.collect()

if DEVICE.type == "cuda":

    torch.cuda.empty_cache()


# ============================================================
# 25. TRAINING
# ============================================================

print("\n" + "=" * 80)
print("STARTING EFFICIENTNET-B0 TRAINING")
print("=" * 80)

print(
    "Objective:"
)

print(
    "8 frame logits -> mean -> video CE"
)

print(
    "Effective batch:",
    VIDEO_BATCH_SIZE * ACCUM_STEPS
)

print(
    "Total epochs:",
    NUM_EPOCHS
)

print("=" * 80)


history = []

best_top1 = -1.0

best_epoch = -1

training_start = time.time()


# ============================================================
# Epoch loop
# ============================================================

for epoch in range(
    1,
    NUM_EPOCHS + 1
):


    epoch_start = time.time()


    # --------------------------------------------------------
    # Training mode
    # --------------------------------------------------------

    model.train()


    # --------------------------------------------------------
    # Epoch 2:
    # Unfreeze backbone
    # --------------------------------------------------------

    if epoch == 2:

        print("\n" + "=" * 80)
        print("UNFREEZING EFFICIENTNET BACKBONE")
        print("=" * 80)

        set_backbone_trainable(
            model,
            True
        )

        print(
            "Trainable parameters:",
            f"{count_trainable_parameters(model):,}"
        )


    # --------------------------------------------------------
    # Small-batch BatchNorm handling
    #
    # Keep pretrained BN running statistics fixed because
    # actual forward micro-batch is only 1 frame.
    #
    # BN affine parameters remain trainable.
    # --------------------------------------------------------

    if epoch >= 2:

        for module in model.modules():

            if isinstance(
                module,
                nn.BatchNorm2d
            ):

                module.eval()


    # --------------------------------------------------------
    # Metrics
    # --------------------------------------------------------

    running_loss = 0.0

    total_samples = 0

    total_top1 = 0

    total_top5 = 0


    # --------------------------------------------------------
    # Gradient accumulation
    # --------------------------------------------------------

    accumulated_videos = 0

    optimizer.zero_grad(
        set_to_none=True
    )


    # ========================================================
    # Batch loop
    # ========================================================

    for batch_idx, (
        videos,
        labels
    ) in enumerate(
        train_loader,
        start=1
    ):


        # ----------------------------------------------------
        # Move to GPU
        # ----------------------------------------------------

        videos = videos.to(
            DEVICE,
            non_blocking=True
        )

        labels = labels.to(
            DEVICE,
            non_blocking=True
        )


        B = videos.size(0)


        # ====================================================
        # Process the 8 frames
        # ====================================================

        frame_logits = []


        for frame_start in range(
            0,
            NUM_FRAMES,
            FRAME_MICRO_BATCH
        ):


            frame_end = min(
                frame_start
                + FRAME_MICRO_BATCH,

                NUM_FRAMES
            )


            # ------------------------------------------------
            # [B, micro, C, H, W]
            # ------------------------------------------------

            frame_batch = videos[
                :,
                frame_start:frame_end
            ]


            # ------------------------------------------------
            # [B*micro, C, H, W]
            # ------------------------------------------------

            frame_batch = frame_batch.reshape(
                -1,
                3,
                IMAGE_SIZE,
                IMAGE_SIZE
            )


            # ------------------------------------------------
            # Forward
            # ------------------------------------------------

            with torch.cuda.amp.autocast(
                enabled=(
                    USE_AMP
                    and DEVICE.type == "cuda"
                )
            ):

                logits = model(
                    frame_batch
                )


            frame_logits.append(
                logits
            )


        # ====================================================
        # [B, 8, 4764]
        # ====================================================

        frame_logits = torch.cat(
            frame_logits,
            dim=0
        )


        frame_logits = frame_logits.reshape(
            B,
            NUM_FRAMES,
            NUM_CLASSES
        )


        # ====================================================
        # EXACT VIDEO-LEVEL OBJECTIVE
        #
        # Mean logits across the 8 frames.
        # ====================================================

        video_logits = frame_logits.float().mean(
            dim=1
        )


        # ====================================================
        # ONE CE LOSS PER VIDEO
        # ====================================================

        loss = F.cross_entropy(
            video_logits,
            labels
        )


        # ====================================================
        # BACKWARD
        #
        # No optimizer step here unless accumulation boundary
        # is reached.
        # ====================================================

        scaler.scale(
            loss
        ).backward()


        # ====================================================
        # Training metrics
        #
        # Calculated from the SAME forward pass.
        # No second forward pass.
        # ====================================================

        batch_top1, batch_top5 = topk_correct(
            video_logits.detach(),
            labels,
            k=5
        )


        running_loss += (
            loss.item() * B
        )

        total_top1 += batch_top1

        total_top5 += batch_top5

        total_samples += B


        # ====================================================
        # VIDEO-LEVEL gradient accumulation
        # ====================================================

        accumulated_videos += B


        should_step = (
            accumulated_videos
            >= (
                VIDEO_BATCH_SIZE
                * ACCUM_STEPS
            )
        )


        # Last partial accumulation group
        if batch_idx == len(
            train_loader
        ):

            should_step = True


        # ====================================================
        # OPTIMIZER STEP
        # ====================================================

        if should_step:


            # ------------------------------------------------
            # Unscale before gradient manipulation
            # ------------------------------------------------

            scaler.unscale_(
                optimizer
            )


            # ------------------------------------------------
            # Average gradients over accumulated videos
            # ------------------------------------------------

            for parameter in model.parameters():

                if parameter.grad is not None:

                    parameter.grad.div_(
                        accumulated_videos
                    )


            # ------------------------------------------------
            # Gradient clipping
            # ------------------------------------------------

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                GRAD_CLIP
            )


            # ------------------------------------------------
            # Optimizer update
            # ------------------------------------------------

            scaler.step(
                optimizer
            )

            scaler.update()


            # ------------------------------------------------
            # Reset
            # ------------------------------------------------

            optimizer.zero_grad(
                set_to_none=True
            )

            accumulated_videos = 0


        # ====================================================
        # Cleanup
        # ====================================================

        del frame_logits
        del video_logits
        del videos
        del labels


        # ====================================================
        # Progress
        # ====================================================

        if (
            batch_idx == 1
            or
            batch_idx % 100 == 0
            or
            batch_idx == len(
                train_loader
            )
        ):

            current_loss = (
                running_loss
                / max(
                    total_samples,
                    1
                )
            )

            current_top1 = (
                total_top1
                / max(
                    total_samples,
                    1
                )
                * 100.0
            )

            current_top5 = (
                total_top5
                / max(
                    total_samples,
                    1
                )
                * 100.0
            )


            print(
                f"Epoch {epoch:02d}/{NUM_EPOCHS} | "
                f"Batch {batch_idx:04d}/{len(train_loader)} | "
                f"Loss: {current_loss:.4f} | "
                f"Top1: {current_top1:.2f}% | "
                f"Top5: {current_top5:.2f}%"
            )


    # ========================================================
    # Epoch metrics
    # ========================================================

    epoch_loss = (
        running_loss
        / max(
            total_samples,
            1
        )
    )

    epoch_top1 = (
        total_top1
        / max(
            total_samples,
            1
        )
        * 100.0
    )

    epoch_top5 = (
        total_top5
        / max(
            total_samples,
            1
        )
        * 100.0
    )


    # ========================================================
    # Scheduler
    # ========================================================

    scheduler.step()


    # ========================================================
    # Learning rates
    # ========================================================

    backbone_lr_now = (
        optimizer.param_groups[0]["lr"]
    )

    classifier_lr_now = (
        optimizer.param_groups[1]["lr"]
    )


    # ========================================================
    # Timing
    # ========================================================

    epoch_time = (
        time.time()
        - epoch_start
    ) / 60.0


    # ========================================================
    # History
    # ========================================================

    record = {

        "epoch":
            epoch,

        "loss":
            epoch_loss,

        "train_top1":
            epoch_top1,

        "train_top5":
            epoch_top5,

        "backbone_lr":
            backbone_lr_now,

        "classifier_lr":
            classifier_lr_now,

        "epoch_time_minutes":
            epoch_time,

        "trainable_parameters":
            count_trainable_parameters(
                model
            )
    }


    history.append(
        record
    )


    history_df = pd.DataFrame(
        history
    )


    history_df.to_csv(
        HISTORY_PATH,
        index=False
    )


    # ========================================================
    # Best checkpoint
    # ========================================================

    if epoch_top1 > best_top1:

        best_top1 = epoch_top1

        best_epoch = epoch


        best_checkpoint = {

            "epoch":
                epoch,

            "model_state_dict":
                model.state_dict(),

            "optimizer_state_dict":
                optimizer.state_dict(),

            "scheduler_state_dict":
                scheduler.state_dict(),

            "scaler_state_dict":
                scaler.state_dict(),

            "best_train_top1":
                best_top1,

            "best_train_top5":
                epoch_top5,

            "config":
                config
        }


        torch.save(
            best_checkpoint,
            BEST_MODEL_PATH
        )


        best_marker = " * BEST"


    else:

        best_marker = ""


    # ========================================================
    # Epoch summary
    # ========================================================

    print("\n" + "-" * 80)

    print(
        f"Epoch {epoch:02d}/{NUM_EPOCHS}"
    )

    print(
        f"Loss: {epoch_loss:.4f}"
    )

    print(
        f"Train Top-1: {epoch_top1:.2f}%"
    )

    print(
        f"Train Top-5: {epoch_top5:.2f}%"
    )

    print(
        f"Backbone LR: {backbone_lr_now:.2e}"
    )

    print(
        f"Classifier LR: {classifier_lr_now:.2e}"
    )

    print(
        f"Time: {epoch_time:.2f} min"
    )

    print(
        f"Best Epoch: {best_epoch}"
        f"{best_marker}"
    )

    print("-" * 80)


    # ========================================================
    # GPU memory
    # ========================================================

    if DEVICE.type == "cuda":

        allocated = (
            torch.cuda.memory_allocated()
            / 1024**3
        )

        reserved = (
            torch.cuda.memory_reserved()
            / 1024**3
        )

        print(
            f"GPU allocated: "
            f"{allocated:.2f} GB"
        )

        print(
            f"GPU reserved: "
            f"{reserved:.2f} GB"
        )


    # ========================================================
    # Cleanup
    # ========================================================

    gc.collect()

    if DEVICE.type == "cuda":

        torch.cuda.empty_cache()


# ============================================================
# 26. Final checkpoint
# ============================================================

final_checkpoint = {

    "epoch":
        NUM_EPOCHS,

    "model_state_dict":
        model.state_dict(),

    "optimizer_state_dict":
        optimizer.state_dict(),

    "scheduler_state_dict":
        scheduler.state_dict(),

    "scaler_state_dict":
        scaler.state_dict(),

    "best_train_top1":
        best_top1,

    "best_epoch":
        best_epoch,

    "final_train_top1":
        history[-1]["train_top1"],

    "final_train_top5":
        history[-1]["train_top5"],

    "config":
        config
}


torch.save(
    final_checkpoint,
    FINAL_MODEL_PATH
)


# ============================================================
# 27. Final history
# ============================================================

history_df = pd.DataFrame(
    history
)

history_df.to_csv(
    HISTORY_PATH,
    index=False
)


# ============================================================
# 28. Final report
# ============================================================

total_training_time = (
    time.time()
    - training_start
) / 60.0


print("\n")
print("=" * 80)
print("EFFICIENTNET-B0 TRAINING COMPLETE")
print("=" * 80)

print(
    f"Best epoch: {best_epoch}"
)

print(
    f"Best Train Top-1: "
    f"{best_top1:.2f}%"
)

print(
    f"Best Train Top-5: "
    f"{history[best_epoch - 1]['train_top5']:.2f}%"
)

print(
    f"Final Train Top-1: "
    f"{history[-1]['train_top1']:.2f}%"
)

print(
    f"Final Train Top-5: "
    f"{history[-1]['train_top5']:.2f}%"
)

print(
    f"Total training time: "
    f"{total_training_time:.2f} minutes"
)


print("\nArtifacts:")

print(
    "\nBEST:"
)

print(
    BEST_MODEL_PATH
)

print(
    "\nFINAL:"
)

print(
    FINAL_MODEL_PATH
)

print(
    "\nHISTORY:"
)

print(
    HISTORY_PATH
)

print(
    "\nCONFIG:"
)

print(
    CONFIG_PATH
)


print("\n" + "=" * 80)
print("SCIENTIFIC INTERPRETATION")
print("=" * 80)

print(
    "Training metrics are convergence diagnostics only."
)

print(
    "They are NOT independent CISLR benchmark results."
)

print(
    "Official evaluation remains prototype -> test retrieval."
)

print(
    "The training objective is:"
)

print(
    "CE(mean(8 frame logits), video label)"
)

print("=" * 80)

DEVICE
Device: cuda
GPU: NVIDIA GeForce RTX 4050 Laptop GPU
CUDA: 13.0
Free GPU memory: 5.54 GB
Total GPU memory: 5.64 GB

LOADING FROZEN COMMON POOL
Original pool shape: (4417, 8)

Status counts:
status
OK    4417
Name: count, dtype: int64

Readable videos: 4417

VERIFYING CACHE
All 4417 cache files exist.

DATALOADER
Dataset videos: 4417
DataLoader batches: 4417
Video batch: 1
Frames/video: 8
Frame micro-batch: 1
Accumulation steps: 16
Effective batch: 16

VALIDATING DATA PIPELINE
Direct dataset shape: torch.Size([8, 3, 224, 224])
Direct label: 0
DataLoader video shape: torch.Size([1, 8, 3, 224, 224])
DataLoader label shape: torch.Size([1])

BUILDING EFFICIENTNET-B0
Classifier input: 1280
Classifier output: 4764

Total parameters: 10,110,232
Epoch 1:
Backbone: FROZEN
Classifier: TRAINABLE

Configuration saved:
/home/dipshikha/Music/cao/CISLR_RESEARCH/audit/stage4_4_training/efficientnet_b0_v2_exact_meanlogit/efficientnet_b0_stage4_4_v2_exact_meanlogit_config.json


/tmp/ipykernel_5708/831695830.py:1030: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(
/tmp/ipykernel_5708/831695830.py:1446: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(



STARTING EFFICIENTNET-B0 TRAINING
Objective:
8 frame logits -> mean -> video CE
Effective batch: 16
Total epochs: 25
Epoch 01/25 | Batch 0001/4417 | Loss: 8.5760 | Top1: 0.00% | Top5: 0.00%
Epoch 01/25 | Batch 0100/4417 | Loss: 8.4853 | Top1: 0.00% | Top5: 0.00%
Epoch 01/25 | Batch 0200/4417 | Loss: 8.4996 | Top1: 0.00% | Top5: 0.00%
Epoch 01/25 | Batch 0300/4417 | Loss: 8.5416 | Top1: 0.00% | Top5: 0.00%
Epoch 01/25 | Batch 0400/4417 | Loss: 8.6012 | Top1: 0.00% | Top5: 0.00%
Epoch 01/25 | Batch 0500/4417 | Loss: 8.6879 | Top1: 0.00% | Top5: 0.00%
Epoch 01/25 | Batch 0600/4417 | Loss: 8.7907 | Top1: 0.00% | Top5: 0.00%
Epoch 01/25 | Batch 0700/4417 | Loss: 8.9058 | Top1: 0.00% | Top5: 0.00%
Epoch 01/25 | Batch 0800/4417 | Loss: 9.0187 | Top1: 0.00% | Top5: 0.00%
Epoch 01/25 | Batch 0900/4417 | Loss: 9.1282 | Top1: 0.00% | Top5: 0.00%
Epoch 01/25 | Batch 1000/4417 | Loss: 9.2272 | Top1: 0.00% | Top5: 0.00%
Epoch 01/25 | Batch 1100/4417 | Loss: 9.3205 | Top1: 0.00% | Top5: 0.00%
Epoch 

/tmp/ipykernel_5708/831695830.py:1446: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(


Epoch 02/25 | Batch 0001/4417 | Loss: 8.1655 | Top1: 0.00% | Top5: 0.00%
Epoch 02/25 | Batch 0100/4417 | Loss: 8.4948 | Top1: 0.00% | Top5: 2.00%
Epoch 02/25 | Batch 0200/4417 | Loss: 8.4735 | Top1: 0.00% | Top5: 1.00%
Epoch 02/25 | Batch 0300/4417 | Loss: 8.4799 | Top1: 0.00% | Top5: 0.67%
Epoch 02/25 | Batch 0400/4417 | Loss: 8.4667 | Top1: 0.00% | Top5: 0.50%
Epoch 02/25 | Batch 0500/4417 | Loss: 8.4673 | Top1: 0.00% | Top5: 0.40%
Epoch 02/25 | Batch 0600/4417 | Loss: 8.4669 | Top1: 0.00% | Top5: 0.50%
Epoch 02/25 | Batch 0700/4417 | Loss: 8.4615 | Top1: 0.00% | Top5: 0.43%
Epoch 02/25 | Batch 0800/4417 | Loss: 8.4603 | Top1: 0.00% | Top5: 0.38%
Epoch 02/25 | Batch 0900/4417 | Loss: 8.4614 | Top1: 0.00% | Top5: 0.33%
Epoch 02/25 | Batch 1000/4417 | Loss: 8.4575 | Top1: 0.00% | Top5: 0.30%
Epoch 02/25 | Batch 1100/4417 | Loss: 8.4619 | Top1: 0.00% | Top5: 0.27%
Epoch 02/25 | Batch 1200/4417 | Loss: 8.4638 | Top1: 0.00% | Top5: 0.25%
Epoch 02/25 | Batch 1300/4417 | Loss: 8.4615 | Top1

...

Epoch 23/25 | Batch 0001/4417 | Loss: 0.0002 | Top1: 100.00% | Top5: 100.00%
Epoch 23/25 | Batch 0100/4417 | Loss: 0.1845 | Top1: 99.00% | Top5: 100.00%
Epoch 23/25 | Batch 0200/4417 | Loss: 0.1865 | Top1: 99.00% | Top5: 100.00%
Epoch 23/25 | Batch 0300/4417 | Loss: 0.1881 | Top1: 99.00% | Top5: 100.00%
Epoch 23/25 | Batch 0400/4417 | Loss: 0.2008 | Top1: 98.75% | Top5: 100.00%
Epoch 23/25 | Batch 0500/4417 | Loss: 0.1955 | Top1: 98.80% | Top5: 100.00%
Epoch 23/25 | Batch 0600/4417 | Loss: 0.1863 | Top1: 99.00% | Top5: 100.00%
Epoch 23/25 | Batch 0700/4417 | Loss: 0.1999 | Top1: 98.86% | Top5: 100.00%
Epoch 23/25 | Batch 0800/4417 | Loss: 0.2065 | Top1: 98.62% | Top5: 100.00%
Epoch 23/25 | Batch 0900/4417 | Loss: 0.2048 | Top1: 98.56% | Top5: 100.00%
Epoch 23/25 | Batch 1000/4417 | Loss: 0.2185 | Top1: 98.40% | Top5: 100.00%
Epoch 23/25 | Batch 1100/4417 | Loss: 0.2151 | Top1: 98.36% | Top5: 100.00%
Epoch 23/25 | Batch 1200/4417 | Loss: 0.2188 | Top1: 98.33% | Top5: 100.00%
Epoch 23/25

/tmp/ipykernel_5708/831695830.py:1446: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(


Epoch 24/25 | Batch 0001/4417 | Loss: 1.2455 | Top1: 100.00% | Top5: 100.00%
Epoch 24/25 | Batch 0100/4417 | Loss: 0.2236 | Top1: 98.00% | Top5: 100.00%
Epoch 24/25 | Batch 0200/4417 | Loss: 0.1782 | Top1: 99.00% | Top5: 100.00%
Epoch 24/25 | Batch 0300/4417 | Loss: 0.2138 | Top1: 97.67% | Top5: 99.67%
Epoch 24/25 | Batch 0400/4417 | Loss: 0.2092 | Top1: 97.75% | Top5: 99.75%
Epoch 24/25 | Batch 0500/4417 | Loss: 0.2010 | Top1: 98.20% | Top5: 99.80%
Epoch 24/25 | Batch 0600/4417 | Loss: 0.1988 | Top1: 98.17% | Top5: 99.67%
Epoch 24/25 | Batch 0700/4417 | Loss: 0.2006 | Top1: 98.43% | Top5: 99.71%
Epoch 24/25 | Batch 0800/4417 | Loss: 0.1962 | Top1: 98.62% | Top5: 99.75%
Epoch 24/25 | Batch 0900/4417 | Loss: 0.1961 | Top1: 98.44% | Top5: 99.78%
Epoch 24/25 | Batch 1000/4417 | Loss: 0.1943 | Top1: 98.60% | Top5: 99.80%
Epoch 24/25 | Batch 1100/4417 | Loss: 0.1957 | Top1: 98.45% | Top5: 99.82%
Epoch 24/25 | Batch 1200/4417 | Loss: 0.1917 | Top1: 98.58% | Top5: 99.83%
Epoch 24/25 | Batch 1

/tmp/ipykernel_5708/831695830.py:1446: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(


Epoch 25/25 | Batch 0001/4417 | Loss: 0.0010 | Top1: 100.00% | Top5: 100.00%
Epoch 25/25 | Batch 0100/4417 | Loss: 0.1906 | Top1: 99.00% | Top5: 100.00%
Epoch 25/25 | Batch 0200/4417 | Loss: 0.1875 | Top1: 98.00% | Top5: 100.00%
Epoch 25/25 | Batch 0300/4417 | Loss: 0.1723 | Top1: 98.67% | Top5: 100.00%
Epoch 25/25 | Batch 0400/4417 | Loss: 0.1966 | Top1: 98.25% | Top5: 99.75%
Epoch 25/25 | Batch 0500/4417 | Loss: 0.1893 | Top1: 98.40% | Top5: 99.80%
Epoch 25/25 | Batch 0600/4417 | Loss: 0.1907 | Top1: 97.83% | Top5: 99.83%
Epoch 25/25 | Batch 0700/4417 | Loss: 0.1795 | Top1: 98.14% | Top5: 99.86%
Epoch 25/25 | Batch 0800/4417 | Loss: 0.1843 | Top1: 98.12% | Top5: 99.88%
Epoch 25/25 | Batch 0900/4417 | Loss: 0.1792 | Top1: 98.33% | Top5: 99.89%
Epoch 25/25 | Batch 1000/4417 | Loss: 0.1768 | Top1: 98.40% | Top5: 99.90%
Epoch 25/25 | Batch 1100/4417 | Loss: 0.1749 | Top1: 98.36% | Top5: 99.91%
Epoch 25/25 | Batch 1200/4417 | Loss: 0.1817 | Top1: 98.25% | Top5: 99.92%
Epoch 25/25 | Batch 